# OceanWatch Analytics · Entrega 1

Lakehouse del tráfico marítimo con datos AIS de NOAA (1–7 junio 2023). Decisiones técnicas y detalles en el `README.md` del repositorio.

# 1. Ingesta

### 1.0 Configuración

In [0]:
import hashlib
import json
import os
import io
import shutil
import time
import urllib.error
import urllib.request
import zipfile
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

CATALOG = "ocean_watch"
SCHEMA = "raw"
VOLUME = "ais_raw"
TABLE = f"{CATALOG}.{SCHEMA}.ais"

YEAR, MONTH = 2023, 6
DAYS = list(range(1, 8))
BASE_URL = "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{year}/{stem}.zip"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_DIR = f"{VOLUME_ROOT}/csv"
MANIFEST_DIR = f"{VOLUME_ROOT}/_manifest"
LOCAL_TMP = "/tmp/ais_download"

MAX_RETRIES = 5
BACKOFF_BASE = 10
TIMEOUT = 120
CHUNK = 8 * 1024 * 1024

EXPECTED_COLUMNS = [
    "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
    "VesselName", "IMO", "CallSign", "VesselType", "Status",
    "Length", "Width", "Draft", "Cargo", "TransceiverClass",
]

spark.conf.set("spark.sql.session.timeZone", "UTC")


def file_stem(day: int) -> str:
    return f"AIS_{YEAR}_{MONTH:02d}_{day:02d}"


def humano(n: float) -> str:
    for unidad in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:,.1f} {unidad}"
        n /= 1024
    return f"{n:,.1f} TB"

### 1.1 Catálogo, esquema y Volume

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
    COMMENT 'OceanWatch Analytics: lakehouse del trafico maritimo (MINE 4213, 2026-20)'
""")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}
    COMMENT 'Capa raw: datos AIS tal como se publican en NOAA Marine Cadastre, sin limpieza'
""")
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}
    COMMENT 'Archivos CSV diarios de AIS descomprimidos y manifiestos de ingesta'
""")

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

### 1.2 Descarga, verificación de integridad y descompresión en el Volume

In [0]:
class IntegrityError(Exception):
    pass


def verify_zip(zip_path: str, expected_member: str) -> zipfile.ZipInfo:
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        if names != [expected_member]:
            raise IntegrityError(f"Contenido inesperado en el zip: {names}")
        bad = zf.testzip()
        if bad is not None:
            raise IntegrityError(f"CRC invalido en {bad}")
        return zf.getinfo(expected_member)


def download_with_retry(url: str, dest: str, expected_member: str) -> dict:
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"  Intento {attempt}/{MAX_RETRIES}: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "ocean-watch-analytics/1.0"})
            sha = hashlib.sha256()
            with urllib.request.urlopen(req, timeout=TIMEOUT) as resp, open(dest, "wb") as f:
                content_length = resp.headers.get("Content-Length")
                etag = resp.headers.get("ETag")
                while chunk := resp.read(CHUNK):
                    f.write(chunk)
                    sha.update(chunk)

            received = os.path.getsize(dest)
            if content_length is not None and received != int(content_length):
                raise IntegrityError(f"Recibidos {received} bytes, se esperaban {content_length}")
            info = verify_zip(dest, expected_member)

            print(f"  OK: {humano(received)}, CRC verificado")
            return {
                "url": url,
                "etag": etag,
                "zip_bytes": received,
                "zip_sha256": sha.hexdigest(),
                "csv_bytes": info.file_size,
                "csv_crc32": f"{info.CRC:08x}",
                "attempts": attempt,
            }
        except urllib.error.HTTPError as exc:
            if 400 <= exc.code < 500 and exc.code != 429:
                raise
            last_exc = exc
        except (urllib.error.URLError, TimeoutError, OSError, IntegrityError, zipfile.BadZipFile) as exc:
            last_exc = exc

        print(f"  Fallo el intento {attempt}: {last_exc!r}")
        if attempt < MAX_RETRIES:
            wait = BACKOFF_BASE * 2 ** (attempt - 1)
            print(f"  Reintentando en {wait} s")
            time.sleep(wait)

    raise RuntimeError(f"No se pudo descargar {url} tras {MAX_RETRIES} intentos") from last_exc


def extract_to_volume(zip_path: str, member: str, dest_csv: str, expected_bytes: int) -> None:
    with zipfile.ZipFile(zip_path) as zf, zf.open(member) as src, open(dest_csv, "wb") as dst:
        shutil.copyfileobj(src, dst, CHUNK)
    written = os.path.getsize(dest_csv)
    if written != expected_bytes:
        raise IntegrityError(f"{dest_csv}: escritos {written} bytes, se esperaban {expected_bytes}")


def ingest_day(day: int) -> dict:
    stem = file_stem(day)
    url = BASE_URL.format(year=YEAR, stem=stem)
    csv_name = f"{stem}.csv"
    csv_path = f"{CSV_DIR}/{csv_name}"
    manifest_path = f"{MANIFEST_DIR}/{stem}.json"

    if os.path.exists(manifest_path) and os.path.exists(csv_path):
        with open(manifest_path) as f:
            manifest = json.load(f)
        if os.path.getsize(csv_path) == manifest["csv_bytes"]:
            print(f"  Ya ingerido y verificado, se omite ({humano(manifest['csv_bytes'])})")
            return {**manifest, "status": "skipped"}

    os.makedirs(LOCAL_TMP, exist_ok=True)
    zip_path = f"{LOCAL_TMP}/{stem}.zip"
    try:
        meta = download_with_retry(url, zip_path, csv_name)
        extract_to_volume(zip_path, csv_name, csv_path, meta["csv_bytes"])
    finally:
        if os.path.exists(zip_path):
            os.remove(zip_path)
    print(f"  Extraido en {csv_path} ({humano(meta['csv_bytes'])})")

    manifest = {
        "day": f"{YEAR}-{MONTH:02d}-{day:02d}",
        "file": csv_name,
        **meta,
        "ingested_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    return {**manifest, "status": "downloaded"}

In [0]:
results, failures = [], {}
for day in DAYS:
    print(f"\n--- {file_stem(day)} ---")
    try:
        results.append(ingest_day(day))
    except Exception as exc:
        failures[file_stem(day)] = repr(exc)
        print(f"  ERROR: {exc!r}")

if results:
    display(pd.DataFrame(results)[
        ["day", "status", "attempts", "zip_bytes", "csv_bytes", "csv_crc32", "zip_sha256", "ingested_at"]
    ])
if failures:
    raise RuntimeError(f"Fallaron {len(failures)} dias: {failures}")

total = 0
for fname in sorted(os.listdir(CSV_DIR)):
    size = os.path.getsize(f"{CSV_DIR}/{fname}")
    total += size
    print(f"{fname}  {humano(size)}")
print(f"Total en el Volume: {humano(total)}")


--- AIS_2023_06_01 ---
  Intento 1/5: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_01.zip
  OK: 328.0 MB, CRC verificado
  Extraido en /Volumes/ocean_watch/raw/ais_raw/csv/AIS_2023_06_01.csv (899.7 MB)

--- AIS_2023_06_02 ---
  Intento 1/5: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_02.zip
  OK: 337.7 MB, CRC verificado
  Extraido en /Volumes/ocean_watch/raw/ais_raw/csv/AIS_2023_06_02.csv (924.5 MB)

--- AIS_2023_06_03 ---
  Intento 1/5: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_03.zip
  OK: 297.8 MB, CRC verificado
  Extraido en /Volumes/ocean_watch/raw/ais_raw/csv/AIS_2023_06_03.csv (820.6 MB)

--- AIS_2023_06_04 ---
  Intento 1/5: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_04.zip
  OK: 317.1 MB, CRC verificado
  Extraido en /Volumes/ocean_watch/raw/ais_raw/csv/AIS_2023_06_04.csv (871.0 MB)

--- AIS_2023_06_05 ---
  Intento 1/5: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_20

day,status,attempts,zip_bytes,csv_bytes,csv_crc32,zip_sha256,ingested_at
2023-06-01,downloaded,1,343949892,943431681,ff1084db,e6287c47f9659fd3bc6c0456579e93f07e3ec099c347514757a86413d0c99d6a,2026-09-25T13:06:17+00:00
2023-06-02,downloaded,1,354145683,969444078,fa5144f1,9f01ff4c3a21240c41ed0ca64a945be7fb8bee0f5bff049dc5c3e727aa02b8ac,2026-09-25T13:06:43+00:00
2023-06-03,downloaded,1,312248025,860511883,784de9af,070ccec839de290cba810308753b56d2bac6839a9fd076df6ab089b525e046c8,2026-09-25T13:07:08+00:00
2023-06-04,downloaded,1,332457486,913283650,d46c5384,396efd491ea992aded336ba85bfed1cd8079baf294fb1fadf215e69d40ed42a0,2026-09-25T13:07:48+00:00
2023-06-05,downloaded,1,335658758,922808494,be9a1d9c,00899e6450a4cc3170059ed990c94e060257e86f22edd5c6683f282e5ce7f4c8,2026-09-25T13:08:27+00:00
2023-06-06,downloaded,1,335300028,918827336,35ee2375,c8b4acd430a675fbffe3665d9ed9843ce0696c947e4d35411a504ad2ffc595f4,2026-09-25T13:08:52+00:00
2023-06-07,downloaded,1,347918296,953614322,93ff4645,03b9b5ddb1247f08a4c3bab738c7b6c569c93094f82e8830cfea4853f9a5bed7,2026-09-25T13:09:21+00:00


AIS_2023_06_01.csv  899.7 MB
AIS_2023_06_02.csv  924.5 MB
AIS_2023_06_03.csv  820.6 MB
AIS_2023_06_04.csv  871.0 MB
AIS_2023_06_05.csv  880.1 MB
AIS_2023_06_06.csv  876.3 MB
AIS_2023_06_07.csv  909.4 MB
Total en el Volume: 6.0 GB


### 1.3 Validación del encabezado

In [0]:
for day in DAYS:
    path = f"{CSV_DIR}/{file_stem(day)}.csv"
    with open(path, encoding="utf-8") as f:
        header = f.readline().strip().lstrip("\ufeff").split(",")
    assert header == EXPECTED_COLUMNS, f"{path}: encabezado inesperado {header}"
print(f"Encabezado correcto en los {len(DAYS)} archivos")

Encabezado correcto en los 7 archivos


### 1.4 Lectura con esquema explícito y carga en `ocean_watch.raw.ais`

Este schema explicito fue tomado directamente de la documentación que está disponible en la página del dataset. Las definiciones y tipos de datos que se exponen salen directamente del diccionario de datos que se proporciona.

Se añadió una columna _corruptrecord cuando alguno de los datos no se puede convertir de manera explicita.

In [0]:
ais_schema = StructType([
    StructField("MMSI",             StringType(),    nullable=True),
    StructField("BaseDateTime",     TimestampType(), nullable=True),
    StructField("LAT",              DoubleType(),    nullable=True),
    StructField("LON",              DoubleType(),    nullable=True),
    StructField("SOG",              DoubleType(),    nullable=True),
    StructField("COG",              DoubleType(),    nullable=True),
    StructField("Heading",          DoubleType(),    nullable=True),
    StructField("VesselName",       StringType(),    nullable=True),
    StructField("IMO",              StringType(),    nullable=True),
    StructField("CallSign",         StringType(),    nullable=True),
    StructField("VesselType",       IntegerType(),   nullable=True),
    StructField("Status",           IntegerType(),   nullable=True),
    StructField("Length",           DoubleType(),    nullable=True),
    StructField("Width",            DoubleType(),    nullable=True),
    StructField("Draft",            DoubleType(),    nullable=True),
    StructField("Cargo",            IntegerType(),   nullable=True),
    StructField("TransceiverClass", StringType(),    nullable=True),
    StructField("_corrupt_record",  StringType(),    nullable=True),
])
assert [f.name for f in ais_schema.fields[:-1]] == EXPECTED_COLUMNS

raw = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(ais_schema)
    .csv(f"{CSV_DIR}/*.csv")
)

ais = (
    raw
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("day", F.to_date("BaseDateTime"))
    .withColumn("ingested_at", F.current_timestamp())
)

(
    ais.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE)
)

### 1.5 Comentarios y metadatos de la tabla

In [0]:
COLUMN_COMMENTS = {
    "MMSI": "Maritime Mobile Service Identity. Identificador del buque, deberia tener 9 digitos",
    "BaseDateTime": "Fecha y hora UTC del reporte de posicion",
    "LAT": "Latitud en grados decimales, rango valido [-90, 90]",
    "LON": "Longitud en grados decimales, rango valido [-180, 180]",
    "SOG": "Speed over ground: velocidad sobre el fondo en nudos",
    "COG": "Course over ground: rumbo sobre el fondo en grados [0, 360)",
    "Heading": "Rumbo de la proa en grados [0, 359]. 511 significa no disponible",
    "VesselName": "Nombre del buque reportado por el transpondedor",
    "IMO": "Numero IMO con prefijo, por ejemplo IMO9074729. Vacio o IMO0000000 si no se reporta",
    "CallSign": "Indicativo de llamada de radio",
    "VesselType": "Codigo AIS de tipo de buque (ver catalogo de tipos de NOAA)",
    "Status": "Estado de navegacion AIS, codigos 0 a 15",
    "Length": "Eslora del buque en metros",
    "Width": "Manga del buque en metros",
    "Draft": "Calado del buque en metros",
    "Cargo": "Codigo AIS del tipo de carga",
    "TransceiverClass": "Clase del transpondedor AIS: A (buques comerciales) o B (embarcaciones menores)",
    "_corrupt_record": "Linea original del CSV cuando algun campo no se pudo convertir al tipo declarado; null si la fila se leyo bien",
    "source_file": "Archivo CSV de origen dentro del Volume",
    "day": "Fecha UTC derivada de BaseDateTime",
    "ingested_at": "Momento de la carga en la tabla",
}

spark.sql(f"""
    COMMENT ON TABLE {TABLE} IS
    'Posiciones AIS crudas de NOAA Marine Cadastre, 1 al 7 de junio de 2023. Una fila por mensaje de posicion, sin limpieza.'
""")
for column, comment in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {TABLE} ALTER COLUMN `{column}` COMMENT '{comment}'")

spark.sql(f"""
    ALTER TABLE {TABLE} SET TBLPROPERTIES (
        'source' = 'NOAA Marine Cadastre AIS',
        'source_url' = 'https://hub.marinecadastre.gov/pages/vesseltraffic',
        'coverage_start' = '{YEAR}-{MONTH:02d}-{DAYS[0]:02d}',
        'coverage_end' = '{YEAR}-{MONTH:02d}-{DAYS[-1]:02d}',
        'layer' = 'raw'
    )
""")

display(spark.sql(f"DESCRIBE TABLE EXTENDED {TABLE}"))

col_name,data_type,comment
MMSI,string,"Maritime Mobile Service Identity. Identificador del buque, deberia tener 9 digitos"
BaseDateTime,timestamp,Fecha y hora UTC del reporte de posicion
LAT,double,"Latitud en grados decimales, rango valido [-90, 90]"
LON,double,"Longitud en grados decimales, rango valido [-180, 180]"
SOG,double,Speed over ground: velocidad sobre el fondo en nudos
COG,double,"Course over ground: rumbo sobre el fondo en grados [0, 360)"
Heading,double,"Rumbo de la proa en grados [0, 359]. 511 significa no disponible"
VesselName,string,Nombre del buque reportado por el transpondedor
IMO,string,"Numero IMO con prefijo, por ejemplo IMO9074729. Vacio o IMO0000000 si no se reporta"
CallSign,string,Indicativo de llamada de radio


### 1.6 Reconciliación CSV vs tabla

In [0]:
csv_lines = (
    spark.read.text(f"{CSV_DIR}/*.csv")
    .where(F.length("value") > 0)
    .groupBy(F.col("_metadata.file_name").alias("source_file"))
    .agg((F.count("*") - 1).alias("csv_rows"))
)

table_rows = (
    spark.table(TABLE)
    .withColumn(
        "file_day",
        F.to_date(F.regexp_extract("source_file", r"(\d{4}_\d{2}_\d{2})", 1), "yyyy_MM_dd"),
    )
    .groupBy("source_file")
    .agg(
        F.count("*").alias("table_rows"),
        F.count("_corrupt_record").alias("corrupt_rows"),
        F.sum(F.when(F.col("day") != F.col("file_day"), 1).otherwise(0)).alias("rows_outside_file_day"),
        F.min("BaseDateTime").alias("min_ts"),
        F.max("BaseDateTime").alias("max_ts"),
    )
)

reconciliation = (
    csv_lines.join(table_rows, "source_file", "full")
    .withColumn("rows_match", F.coalesce(F.col("csv_rows") == F.col("table_rows"), F.lit(False)))
    .orderBy("source_file")
    .toPandas()
)
display(reconciliation)

print(f"Filas totales en {TABLE}: {reconciliation['table_rows'].sum():,}")
print(f"Filas con _corrupt_record: {reconciliation['corrupt_rows'].sum():,}")
assert len(reconciliation) == len(DAYS), f"Se esperaban {len(DAYS)} archivos, hay {len(reconciliation)}"
assert reconciliation["rows_match"].all(), "Hay archivos cuyo numero de filas no coincide con la tabla"

source_file,csv_rows,table_rows,corrupt_rows,rows_outside_file_day,min_ts,max_ts,rows_match
AIS_2023_06_01.csv,8808904,8808904,0,0,2023-06-01T00:00:00.000Z,2023-06-01T23:59:59.000Z,true
AIS_2023_06_02.csv,9052241,9052241,0,0,2023-06-02T00:00:00.000Z,2023-06-02T23:59:59.000Z,true
AIS_2023_06_03.csv,8036348,8036348,0,0,2023-06-03T00:00:00.000Z,2023-06-03T23:59:59.000Z,true
AIS_2023_06_04.csv,8522645,8522645,0,0,2023-06-04T00:00:00.000Z,2023-06-04T23:59:59.000Z,true
AIS_2023_06_05.csv,8613757,8613757,0,0,2023-06-05T00:00:00.000Z,2023-06-05T23:59:59.000Z,true
AIS_2023_06_06.csv,8587938,8587938,0,0,2023-06-06T00:00:00.000Z,2023-06-06T23:59:59.000Z,true
AIS_2023_06_07.csv,8911726,8911726,0,0,2023-06-07T00:00:00.000Z,2023-06-07T23:59:59.000Z,true


Filas totales en ocean_watch.raw.ais: 60,533,559
Filas con _corrupt_record: 0


In [0]:
display(
    spark.table(TABLE)
    .where("_corrupt_record IS NOT NULL")
    .select("source_file", "_corrupt_record")
    .limit(20)
)

source_file,_corrupt_record


### 1.7 Limpieza de artefactos de la versión anterior (opcional)

In [0]:
# TODO: que significa esto en terminos practicos

In [0]:
CLEANUP_LEGACY = False

legacy_table = "workspace.default.ais"
legacy_files = [f"{VOLUME_ROOT}/ais_2023_06_{day:02d}" for day in DAYS]

print(f"{legacy_table} existe: {spark.catalog.tableExists(legacy_table)}")
print(f"Archivos antiguos en el Volume: {[p for p in legacy_files if os.path.exists(p)]}")

if CLEANUP_LEGACY:
    spark.sql(f"DROP TABLE IF EXISTS {legacy_table}")
    for path in legacy_files:
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

workspace.default.ais existe: False
Archivos antiguos en el Volume: []


### 1.8 Carga del World Port Index

In [0]:
WPI_FILE_ID = "1VyCGCAfFuEK7vB1C9Vq8iPdgBdu-LDM4"
url = f"https://drive.google.com/uc?export=download&id={WPI_FILE_ID}"
req = urllib.request.Request(url, headers={"User-Agent": "ocean-watch-analytics/1.0"})
with urllib.request.urlopen(req, timeout=120) as resp:
    zip_data = resp.read()

os.makedirs("/tmp/wpi", exist_ok=True)
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extract("WPI.mdb", "/tmp/wpi/")
print(f"WPI.mdb descargado: {os.path.getsize('/tmp/wpi/WPI.mdb'):,} bytes")

WPI.mdb descargado: 18,735,104 bytes


In [0]:
%pip install access-parser

from access_parser import AccessParser

db = AccessParser("/tmp/wpi/WPI.mdb")
table = db.get_table("Wpi Data")
table.parse()
import pandas as pd
wpi_pdf = pd.DataFrame(table.parsed_table)
print(f"Puertos en el WPI: {len(wpi_pdf):,}")

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for access-parser: filename=access_parser-0.0.6-py3-none-any.whl size=18016 sha256=8f7acd928ec8dbf20cef5c0b9df2b73fec428a8d9d1e60d439b4b2666946f8d8
  Stored in directory: /home/spark-48225840-a803-4770-868a-cd/.cache/pip/wheels/8c/9f/d3/2e80f9f0d16b66c511bfe0fdbfc287b519c31f1a5d677ae279
Successfully built access-parser
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Puertos en el WPI: 3,630


In [0]:
# Convertir coordenadas de grados+minutos a grados decimales
# LAT = degrees + minutes/60, negativo si hemisferio es S
# LON = degrees + minutes/60, negativo si hemisferio es W
wpi_pdf["LAT"] = wpi_pdf["Latitude_degrees"] + wpi_pdf["Latitude_minutes"] / 60.0
wpi_pdf.loc[wpi_pdf["Latitude_hemisphere"] == "S", "LAT"] *= -1
wpi_pdf["LON"] = wpi_pdf["Longitude_degrees"] + wpi_pdf["Longitude_minutes"] / 60.0
wpi_pdf.loc[wpi_pdf["Longitude_hemisphere"] == "W", "LON"] *= -1

In [0]:
wpi_cols = ["World_port_index_number", "Main_port_name", "Wpi_country_code", "LAT", "LON"]
wpi_df = spark.createDataFrame(wpi_pdf[wpi_cols].rename(columns={
    "World_port_index_number": "port_index",
    "Main_port_name": "port_name",
    "Wpi_country_code": "country_code",
}))

wpi_df.write.mode("overwrite").saveAsTable("ocean_watch.raw.world_port_index")
print(f"Tabla ocean_watch.raw.world_port_index creada con {wpi_df.count():,} puertos")

display(spark.table("ocean_watch.raw.world_port_index").limit(10))

Tabla ocean_watch.raw.world_port_index creada con 3,630 puertos


port_index,port_name,country_code,LAT,LON
19130,KETCHIKAN,US,55.333333333333336,-131.65
19140,WARD COVE,US,55.4,-131.73333333333332
19160,KNUDSON COVE,US,55.46666666666667,-131.8
19170,LORING,US,55.6,-131.63333333333333
19180,YES BAY,US,55.916666666666664,-131.8
19190,UNION BAY,US,55.766666666666666,-132.21666666666667
19200,BURNETT INLET,US,56.06666666666667,-132.46666666666667
19210,WRANGELL,US,56.46666666666667,-132.38333333333333
19220,KASAAN,US,55.53333333333333,-132.4
19230,ROSE INLET,US,54.95,-132.96666666666667


# 2. Exploración y perfilamiento

Esta sección mide el contenido de `ocean_watch.raw.ais` antes de responder las preguntas de negocio de la sección 3. El perfilamiento responde tres preguntas sobre las 60.533.559 filas cargadas.

1. **Volumen.** Cuántas filas y cuántos buques hay por día.
2. **Composición.** Cómo se distribuyen el tipo de buque y las dimensiones declaradas.
3. **Calidad.** Qué proporción de filas rompe cada regla declarada.

Cada bloque de código termina con una ficha de resultado y decisión técnica. La tabla consolidada de 2.6 es el insumo directo de la Entrega 2.

### Convención de conteo

Una fila equivale a una posición AIS. Un buque equivale a un valor distinto de `MMSI`. Un buque con alta frecuencia de transmisión aporta miles de filas al total.

Las métricas de actividad (posiciones por día, velocidad, rumbo) usan la tabla completa. Las métricas de flota (dimensiones, buques únicos) aplican `dropDuplicates(["MMSI"])` antes de agregar. Esta separación evita que los buques que más transmiten desplacen los percentiles de la flota.

### Estado de la sección

| Bloque | Estado |
| --- | --- |
| 2.1 Volumen por día | LISTO |
| 2.2 Tipo de buque | LISTO |
| 2.3 Dimensiones | LISTO |
| 2.4 Reglas de calidad R1 a R9 | LISTO |
| 2.5 Columnas complementarias | LISTO |
| 2.6 Diagnóstico consolidado | LISTO |

El Anexo A, al final del notebook, muestra el costo de una limpieza que descarta toda fila marcada.

In [0]:
# Configuración de la sección 2.
# Todas las celdas siguientes usan estas variables.
from pyspark.sql import functions as F

ais = spark.table("ocean_watch.raw.ais")
TOTAL_FILAS = ais.count()

print(f"Filas en ocean_watch.raw.ais: {TOTAL_FILAS:,}")

Filas en ocean_watch.raw.ais: 60,533,559


### 2.1 Volumen por día

Mide el tamaño de la carga y el número de actores distintos en cada uno de los 7 días.

In [0]:
# Posiciones = filas de la tabla. Buques = valores distintos de MMSI.
# countDistinct entrega el valor exacto. approx_count_distinct entrega el valor aproximado
# y alimenta la comparación de costo de la pregunta 3a, dentro de la misma pasada.
volumen_diario = (
    ais.groupBy("day")
    .agg(
        F.count("*").alias("posiciones"),
        F.countDistinct("MMSI").alias("buques_exacto"),
        F.approx_count_distinct("MMSI").alias("buques_aprox"),
    )
    .orderBy("day")
)

volumen_diario.display()

day,posiciones,buques_exacto,buques_aprox
2023-06-01,8808904,20448,21782
2023-06-02,9052241,21153,23182
2023-06-03,8036348,20505,20877
2023-06-04,8522645,19720,20275
2023-06-05,8613757,19615,20358
2023-06-06,8587938,19717,20288
2023-06-07,8911726,20086,21184


#### Ficha 2.1 · Volumen por día

De lo anterior se desprende que: 
**1. La carga está completa.** La suma de los 7 días da 60.533.559 posiciones. Esa cifra coincide con el conteo de la tabla y con la reconciliación CSV contra tabla de 1.6. Ningún día quedó a medias en la descarga.

**2. El volumen es estable.** El día con más posiciones es el viernes 2 de junio, con 9.052.241. El día con menos es el sábado 3 de junio, con 8.036.348. La diferencia entre los dos extremos es 12,6%. Cada día aporta entre 13,28% y 14,95% del total.

**3. Los buques activos varían menos que las posiciones.** El rango de buques por día va de 19.615 a 21.153, que es una diferencia de 7,8%. Las posiciones por buque se mueven entre 392 y 444. La caída del sábado 3 viene de una menor frecuencia de transmisión por buque, con un número de buques activos que se mantiene.

**4. El aproximado sobreestima entre 1,8% y 9,6%.** `approx_count_distinct` usa HyperLogLog con un error relativo objetivo de 5% por defecto. Los 7 días quedan dentro de ese margen, con una desviación media de 4,7%. El sesgo es siempre positivo en esta corrida.

**Decisión técnica 1. Dos conteos en una sola pasada.** `count`, `countDistinct` y `approx_count_distinct` viajan en el mismo `groupBy`. Spark resuelve el plan con un solo escaneo de la tabla. Tres consultas separadas cuestan tres escaneos de 60,5 millones de filas.

**Decisión técnica 2. Cardinalidad baja, conteo exacto.** El número de buques por día está en el orden de 20.000. `countDistinct` sobre esa cardinalidad cabe en memoria sin riesgo de derrame a disco. La justificación completa está en la pregunta 3a.

**Decisión técnica 3. Insumo de la sección 4.** Las 7 particiones tienen tamaño parejo. Una partición por `day` produce archivos de tamaño similar y evita el sesgo de datos que degrada la lectura en paralelo.

### 2.2 Distribución por tipo de buque

`VesselType` es un código numérico del estándar AIS. El catálogo de referencia es el de MarineCadastre y la guía AIS del USCG.

In [0]:
# Distribución de posiciones por código de VesselType.
# El catálogo de códigos está en:
#   https://coast.noaa.gov/data/marinecadastre/ais/VesselTypeCodes2018.pdf
#   https://www.navcen.uscg.gov/sites/default/files/pdf/AIS/AISGuide.pdf
# La traducción de código a nombre está en la ficha 2.2, para evitar una segunda
# pasada sobre las 60,5 millones de filas.
ais.groupBy("VesselType").count().orderBy("count", ascending=False).display()

VesselType,count
31,16558120
37,14476696
60,4795209
30,4010188
36,3876264
90,3848162
70,3837926
52,1921565
80,1789168
57,1323765


#### Ficha 2.2 · Tipo de buque

**Resultado.** Dos códigos concentran el 51,27% de las posiciones. La distribución tiene una cola larga de 70 códigos con volumen bajo.

| Código | Nombre en el catálogo AIS | Posiciones | % del total |
| --- | --- | --- | --- |
| 31 | Towing (remolcador con remolque) | 16.558.120 | 27,35% |
| 37 | Pleasure craft (embarcación de recreo) | 14.476.696 | 23,92% |
| 60 | Passenger (pasaje, tipo genérico) | 4.795.209 | 7,92% |
| 30 | Fishing (pesca) | 4.010.188 | 6,62% |
| 36 | Sailing (vela) | 3.876.264 | 6,40% |
| 90 | Other type (otro tipo) | 3.848.162 | 6,36% |
| 70 | Cargo (carga) | 3.837.926 | 6,34% |
| 52 | Tug (remolcador) | 1.921.565 | 3,17% |
| 80 | Tanker (tanquero) | 1.789.168 | 2,96% |
| 0 | Not available or no ship | 715.334 | 1,18% |
| null | Sin valor en la columna | 209.863 | 0,35% |

**Códigos con volumen mínimo.** Los códigos 91 (10 filas), 107 (2 filas) y 18 (1 fila) aparecen en el rango reservado a uso regional y a categorías especiales. El volumen es insuficiente para un análisis por separado. Estos códigos entran en el grupo de cola larga.

**Decisión técnica.** El estándar AIS define el código 0 como "not available or no ship". El valor nulo describe la misma situación operativa. Las dos categorías se agrupan bajo el nombre "Sin tipo reportado", con 925.197 filas en total (1,53%). Los análisis posteriores tratan ese grupo como una sola categoría y declaran esa cobertura.

**Límite de la medida.** Esta distribución cuenta posiciones. Un buque que transmite cada 2 segundos aporta mucho más volumen que uno que transmite cada 3 minutos. La distribución de la flota por número de buques se obtiene con la deduplicación por `MMSI` de 2.3.

### 2.3 Dimensiones declaradas (`Length`, `Width`)

`Length` es la eslora en metros. `Width` es la manga en metros. Las dos columnas describen el buque, por lo que el perfil corre sobre buques distintos.

In [0]:
# El perfil de dimensiones describe la flota, por lo que corre sobre buques distintos.
# dropDuplicates(["MMSI"]) conserva una fila por buque.
buques_unicos = ais.select("MMSI", "Length", "Width").dropDuplicates(["MMSI"])

buques_unicos.select(
    F.count("*").alias("total_buques"),
    F.sum(F.col("Length").isNull().cast("long")).alias("nulos"),
    F.min("Length").alias("minimo"),
    F.max("Length").alias("maximo"),
    F.avg("Length").alias("promedio"),
    F.expr("percentile_approx(Length, 0.05)").alias("percentil_5"),
    F.expr("percentile_approx(Length, 0.95)").alias("percentil_95"),
).show()

buques_unicos.select(
    F.count("*").alias("total_buques"),
    F.sum(F.col("Width").isNull().cast("long")).alias("nulos"),
    F.min("Width").alias("minimo"),
    F.max("Width").alias("maximo"),
    F.avg("Width").alias("promedio"),
    F.expr("percentile_approx(Width, 0.05)").alias("percentil_5"),
    F.expr("percentile_approx(Width, 0.95)").alias("percentil_95"),
).show()

+------------+-----+------+------+------------------+-----------+------------+
|total_buques|nulos|minimo|maximo|          promedio|percentil_5|percentil_95|
+------------+-----+------+------+------------------+-----------+------------+
|       31871| 2877|   0.0| 638.0|38.029316410291784|        0.0|       199.0|
+------------+-----+------+------+------------------+-----------+------------+

+------------+-----+------+------+----------------+-----------+------------+
|total_buques|nulos|minimo|maximo|        promedio|percentil_5|percentil_95|
+------------+-----+------+------+----------------+-----------+------------+
|       31871| 6226|   0.0| 126.0|8.64281536361864|        0.0|        32.0|
+------------+-----+------+------+----------------+-----------+------------+



In [0]:
# El mínimo de las dos columnas es 0,0. El valor 0 indica dimensión sin declarar
# dentro del mensaje AIS, y queda separado del conteo de nulos.
# La mediana completa el perfil entre los percentiles 5 y 95.
buques_unicos.select(
    F.sum((F.col("Length") == 0).cast("long")).alias("length_en_cero"),
    F.sum((F.col("Width") == 0).cast("long")).alias("width_en_cero"),
    F.expr("percentile_approx(Length, 0.50)").alias("length_mediana"),
    F.expr("percentile_approx(Width, 0.50)").alias("width_mediana"),
).show()

+--------------+-------------+--------------+-------------+
|length_en_cero|width_en_cero|length_mediana|width_mediana|
+--------------+-------------+--------------+-------------+
|          1700|         1817|          15.0|          5.0|
+--------------+-------------+--------------+-------------+



In [0]:
# Revisión de los valores extremos de Length, con el nombre declarado del buque.
ais.where(F.col("Length") > 450).orderBy("Length", ascending=False).display(10)

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,_corrupt_record,source_file,day,ingested_at
368232140,2023-06-06T22:14:53.000Z,28.40802,-80.6778,0.7,0.0,95.0,LUNA SEA,IMO0000000,WDM7238,37,null,638.0,126.0,null,null,B,null,AIS_2023_06_06.csv,2023-06-06,2026-09-25T13:09:34.912Z
368232140,2023-06-06T22:17:52.000Z,28.40802,-80.6778,0.7,0.0,94.0,LUNA SEA,IMO0000000,WDM7238,37,null,638.0,126.0,null,null,B,null,AIS_2023_06_06.csv,2023-06-06,2026-09-25T13:09:34.912Z
368232140,2023-06-06T22:11:53.000Z,28.40803,-80.6778,0.7,0.0,93.0,LUNA SEA,IMO0000000,WDM7238,37,null,638.0,126.0,null,null,B,null,AIS_2023_06_06.csv,2023-06-06,2026-09-25T13:09:34.912Z
339220000,2023-06-01T05:31:32.000Z,41.00273,-72.29123,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z
339220000,2023-06-01T03:37:17.000Z,41.00273,-72.29123,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z
339220000,2023-06-01T06:19:38.000Z,41.00272,-72.29123,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z
339220000,2023-06-01T00:19:39.000Z,41.00272,-72.29127,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z
339220000,2023-06-01T02:22:09.000Z,41.00272,-72.29125,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z
339220000,2023-06-01T07:28:47.000Z,41.00272,-72.29125,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z
339220000,2023-06-01T10:50:16.000Z,41.0027,-72.29122,0.7,0.0,511.0,7KNOTS105,IMO0000000,6YVG8,37,null,511.0,63.0,null,null,B,null,AIS_2023_06_01.csv,2023-06-01,2026-09-25T13:09:34.912Z


#### Ficha 2.3 · Dimensiones

**Resultado.** La tabla tiene 31.871 buques distintos.

| Columna | Nulos | % | Ceros | % | Sin dato útil | % | Mínimo | Mediana | Promedio | Percentil 95 | Máximo |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `Length` | 2.877 | 9,03% | 1.700 | 5,33% | 4.577 | 14,36% | 0,0 | 15,0 | 38,03 | 199,0 | 638,0 |
| `Width` | 6.226 | 19,54% | 1.817 | 5,70% | 8.043 | 25,24% | 0,0 | 5,0 | 8,64 | 32,0 | 126,0 |

**Lectura 1. La flota es de embarcación pequeña.** La mediana de `Length` es 15,0 metros y la mediana de `Width` es 5,0 metros. La mitad de los buques cabe en esas medidas. Ese perfil concuerda con la distribución de tipos de 2.2, donde el código 37 (embarcación de recreo) aporta el 23,92% de las posiciones.

**Lectura 2. El promedio queda muy por encima de la mediana.** El promedio de `Length` es 38,03 metros frente a una mediana de 15,0. El percentil 95 llega a 199,0 metros. La distribución tiene una cola larga hacia los buques grandes de carga y tanque. Un promedio de eslora describe mal a esta flota, por lo que el reporte usa la mediana y los percentiles.

**Lectura 3. La cobertura real es menor que el conteo de nulos.** `Length` suma 4.577 buques sin dato útil entre nulos y ceros, que es el 14,36% de la flota. `Width` llega al 25,24%. El conteo de nulos por sí solo subestima la falta de dato en 5 puntos porcentuales.

**Hallazgo 1. Valor fuera del rango físico.** El buque "LUNA SEA" (MMSI 368232140) declara `Length` 638,0 y `Width` 126,0, con `VesselType` 37 y transpondedor Clase B. El buque portacontenedores más grande en servicio mide cerca de 400 metros. El valor 638,0 corresponde a un dato erróneo.

**Hallazgo 2. El valor 511 aparece como relleno en `Length`.** El buque "7KNOTS105" (MMSI 339220000) declara `Length` 511,0 y `Heading` 511,0 en la misma fila. El estándar AIS reserva 511 para `Heading` con el significado de no disponible. La aparición del mismo número en la eslora indica un relleno aplicado por el emisor fuera de la especificación.

**Decisión técnica 1. Perfil por buque.** Un perfil sobre las 60,5 millones de filas mide la actividad de transmisión. Los buques que más transmiten desplazan los percentiles hacia sus propias dimensiones. `dropDuplicates(["MMSI"])` reduce la tabla a un valor por buque y entrega el perfil de la flota.

**Decisión técnica 2. Costo de la deduplicación.** `dropDuplicates` sobre `MMSI` provoca un shuffle con 60,5 millones de filas en la entrada y 31.871 en la salida. El perfil corre una sola vez y el resultado alimenta las dos columnas.

**Decisión técnica 3. Límite conocido de la deduplicación.** Un mismo `MMSI` puede declarar dimensiones distintas entre filas. `dropDuplicates` conserva una de esas filas de forma arbitraria. La verificación de consistencia por buque queda registrada como mejora opcional.

**Decisión técnica 4. Cero separado de nulo.** El nulo indica que la columna llegó vacía. El cero indica que el emisor transmitió un valor sin contenido útil. Las dos situaciones se cuentan en columnas distintas.

### 2.4 Reglas de calidad

A continuacion se presentan las reglas de calidad de datos basadas en las reglas de negocio. Cada regla marca una fila con una condición booleana. Una fila puede romper varias reglas al mismo tiempo. Por esa razón la sección entrega un porcentaje por cada regla. Una cifra única de datos sucios ocultaría esa superposición.

#### Tres categorías de hallazgo
- El perfilamiento separa tres categorías.
- Cada ficha declara la categoría que mide su regla. El conteo de valores de no disponible va siempre en una columna propia.

- Varios campos reservan su valor máximo como código de dato no disponible. 

| Categoría | Significado | Ejemplos en este dataset |
| --- | --- | --- |
| Valor de no disponible | El protocolo reserva ese valor para declarar el dato como no disponible | `Heading = 511`, `COG = 360.0`, `SOG = 102.3`, `IMO0000000` |
| Valor ausente | La columna llega nula | `Status`, `Draft`, `Cargo` |
| Valor inválido | El dato existe y rompe un límite físico o de formato | `MMSI` con menos de 9 dígitos, `IMO` con 10 dígitos |

#### Catálogo de reglas

| Regla | Columna | Condición |
| --- | --- | --- |
| R1 | `MMSI` | El identificador no tiene 9 dígitos numéricos |
| R2 | `LAT`, `LON` | La coordenada queda fuera de \[-90, 90\] o \[-180, 180\] |
| R3 | `SOG` | La velocidad supera el umbral de 35 nudos, con 102,3 excluido |
| R4 | `Heading` | El valor es 511, que significa rumbo no disponible |
| R5 | `IMO` | El valor está ausente, trae `IMO0000000` o trae un formato irregular |
| R6 | `_corrupt_record` | La fila no calzó con el esquema al leer el CSV |
| R7 | `BaseDateTime` | La fecha de la fila no coincide con la fecha del archivo de origen |
| R8 | Fila completa | La fila se repite exacta en otra fila |
| R9 | (`MMSI`, `BaseDateTime`) | La llave se repite con posiciones distintas |

#### R1 · `MMSI` con formato distinto de 9 dígitos

In [0]:
# R1. El estándar ITU asigna 9 dígitos a todo identificador MMSI.
# Los tres primeros dígitos son el MID, que identifica el país de matrícula.
MMSI_VALIDO = F.col("MMSI").rlike(r"^\d{9}$")

ais.select(
    F.count("*").alias("total_filas"),
    F.sum(F.col("MMSI").isNull().cast("long")).alias("nulos"),
    F.sum(MMSI_VALIDO.cast("long")).alias("validos_9_digitos"),
    F.sum((~MMSI_VALIDO).cast("long")).alias("invalidos"),
).show()

+-----------+-----+-----------------+---------+
|total_filas|nulos|validos_9_digitos|invalidos|
+-----------+-----+-----------------+---------+
|   60533559|    0|         60483662|    49897|
+-----------+-----+-----------------+---------+



In [0]:
# Evidencia: identificadores que rompen R1, con el nombre declarado del buque.
ais.where(~MMSI_VALIDO).select("MMSI", "VesselName").distinct().show(20, truncate=False)

+----------+------------------+
|MMSI      |VesselName        |
+----------+------------------+
|3126547   |NULL              |
|987       |BUOPY 7 B&J       |
|9110192   |NULL              |
|3669883   |NULL              |
|1         |A                 |
|3126546   |NULL              |
|602       |BUOY 2 E&B 100%   |
|3660489   |NEPTUNE TLP       |
|1072211352|SHAMBHALA         |
|63280801  |3280801 10%       |
|2981331   |JOHN JAMES CHARLES|
|601       |BUOY 1 E&B 97%    |
|3669884   |NULL              |
|69918000  |US GOV VESSEL 87  |
|368926    |US GOVT VESSEL    |
|4566545   |NULL              |
|4566544   |NULL              |
|36892100  |WARSHIP           |
|91800534  |ONWA33NT          |
|4556531   |NULL              |
+----------+------------------+
only showing top 20 rows


#### Ficha R1 · `MMSI`

**Categoría.** Valor inválido de formato.

**Resultado.** 49.897 filas rompen la regla, que es menos del 1% del total. La columna no tiene nulos. El resto de filas cumplen el formato de 9 dígitos.

**Lectura de la evidencia.** Los identificadores cortos ("1", "601", "602", "987") pertenecen a _ayudas a la navegación_. Los nombres asociados lo confirman: "BUOY 1 E&B 97%", "BUOY 2 E&B 100%", "BUOPY 7 B&J". Los identificadores de 7 y 8 dígitos ("3126547", "69918000", "36892100") aparecen junto a nombres como "US GOV VESSEL 87" y "WARSHIP". El identificador "1072211352" tiene 10 dígitos y supera el rango del estándar.

**Decisión técnica 1. Umbral sin excepciones.** La regla aplica el formato del estándar a todas las filas. Las ayudas a la navegación y los buques de gobierno quedan marcados igual que el resto. Esta decisión mantiene una sola definición de identificador válido en todo el proyecto.

**Decisión técnica 2. Marcar sin borrar.** La regla marca la fila y conserva el dato en `ais_raw`. Los análisis con llave de identidad de buque (3a, 3c y 3e) excluyen las filas marcadas y declaran esa exclusión. El efecto sobre los agregados es bajo, porque la proporción es 0,08%.

**Mejora opcional.** El estándar indica una manera de verificar la validez del identificador mediante ala aplicacion de un checksum. Este chequeo se deja explciitamente por fuera del scope de esta entrega pero se tendra en cuenta par ala Entrega 2.

#### R2 · Coordenadas fuera del rango geográfico

In [0]:
# R2. LAT y LON describen una posición sobre el elipsoide WGS84.
# El rango es [-90, 90] para LAT y [-180, 180] para LON.
# El estándar AIS reserva LAT = 91 y LON = 181 para posición no disponible.
# Esos valores quedan fuera de rango, por lo que la misma regla los captura.
COORD_FUERA = (
    F.col("LAT").isNull() | F.col("LON").isNull()
    | (F.col("LAT") < -90) | (F.col("LAT") > 90)
    | (F.col("LON") < -180) | (F.col("LON") > 180)
)

ais.select(
    F.count("*").alias("total_filas"),
    F.sum(F.col("LAT").isNull().cast("long")).alias("lat_nulos"),
    F.sum(F.col("LON").isNull().cast("long")).alias("lon_nulos"),
    F.sum(((F.col("LAT") < -90) | (F.col("LAT") > 90)).cast("long")).alias("lat_fuera_de_rango"),
    F.sum(((F.col("LON") < -180) | (F.col("LON") > 180)).cast("long")).alias("lon_fuera_de_rango"),
).show()

# Evidencia: la tabla debe salir vacía cuando el conteo es cero.
ais.where(
    ((F.col("LAT") < -90) | (F.col("LAT") > 90))
    | ((F.col("LON") < -180) | (F.col("LON") > 180))
).select("MMSI", "LAT", "LON", "BaseDateTime").show(20, truncate=False)

+-----------+---------+---------+------------------+------------------+
|total_filas|lat_nulos|lon_nulos|lat_fuera_de_rango|lon_fuera_de_rango|
+-----------+---------+---------+------------------+------------------+
|   60533559|        0|        0|                 0|                 0|
+-----------+---------+---------+------------------+------------------+

+----+---+---+------------+
|MMSI|LAT|LON|BaseDateTime|
+----+---+---+------------+
+----+---+---+------------+



In [0]:
ais.select(
    F.min("LAT").alias("lat_min"), F.max("LAT").alias("lat_max"),
    F.min("LON").alias("lon_min"), F.max("LON").alias("lon_max"),
).show()

+-------+-------+----------+--------+
|lat_min|lat_max|   lon_min| lon_max|
+-------+-------+----------+--------+
|0.62368|89.6552|-179.95459|146.3172|
+-------+-------+----------+--------+



#### Ficha R2 · Coordenadas

**Categoría.** Valor inválido por límite físico.

**Resultado.** 0 filas quedan fuera de rango. 0 filas traen `LAT` o `LON` nulos. La tabla de evidencia sale vacía, tal como corresponde a un conteo en cero.

**Ausencia de valores de no disponible.** El estándar AIS reserva `LAT = 91` y `LON = 181` para posición no disponible. Los dos valores superan el rango válido. El conteo de fuera de rango en cero demuestra que ninguna fila trae esos valores.

**Caja envolvente real.**

| Medida | Valor |
| --- | --- |
| `lat_min` | 0,62368 |
| `lat_max` | 89,6552 |
| `lon_min` | -179,95459 |
| `lon_max` | 146,3172 |

**Lectura de la caja envolvente.** El rango cubre casi todo el planeta. `lat_max` 89,6552 queda a unos 38 kilómetros del Polo Norte. `lat_min` 0,62368 queda sobre el ecuador. `lon_max` 146,3172 cae en el Pacífico occidental. El dataset de MarineCadastre cubre las aguas de Estados Unidos, por lo que esas posiciones extremas quedan fuera del área esperada.

**Consecuencia para la regla.** La regla de rango global no marca esas filas, porque los valores son válidos como coordenada. Una regla de rango por área de interés sí las marcaría. Esa segunda regla queda registrada como mejora opcional, con la caja de las aguas de Estados Unidos como referencia.

**Decisión técnica 1. Conservar la regla con resultado cero.** El valor cero es evidencia y se reporta en la tabla consolidada. La regla funciona además como control de regresión para cargas futuras.

**Decisión técnica 2. Origen del resultado limpio.** La lectura de 1.4 usa esquema explícito con `DoubleType` para las dos columnas. Las dos condiciones (esquema explícito y recorte del origen) explican la ausencia de coordenadas fuera de rango.


#### R3 · `SOG` sobre el umbral físico

In [0]:
# R3. SOG viaja en el mensaje AIS como entero de 10 bits, en unidades de 0,1 nudos.
# El rango técnico va de 0 a 102,2 nudos. El estándar reserva 102,3 para
# "velocidad no disponible", por lo que ese valor se cuenta por separado.
SOG_NO_DISPONIBLE = 102.3
SOG_MAX = 35.0

SOG_INVALIDO = (
    F.col("SOG").isNotNull()
    & (F.col("SOG") != SOG_NO_DISPONIBLE)
    & ((F.col("SOG") < 0) | (F.col("SOG") > SOG_MAX))
)

ais.select(
    F.count("*").alias("total_filas"),
    F.sum(F.col("SOG").isNull().cast("long")).alias("nulos"),
    F.sum((F.col("SOG") == SOG_NO_DISPONIBLE).cast("long")).alias("no_disponible_102_3"),
    F.sum(SOG_INVALIDO.cast("long")).alias("invalidos"),
    F.max("SOG").alias("maximo"),
    F.expr("percentile_approx(SOG, 0.99)").alias("percentil_99"),
).show()

# Evidencia: dónde se concentran los valores que superan el umbral.
(
    ais.where(SOG_INVALIDO)
    .groupBy("VesselType", "TransceiverClass")
    .agg(F.count("*").alias("filas"), F.max("SOG").alias("sog_maximo"))
    .orderBy(F.desc("filas"))
    .show(10)
)

+-----------+-----+-------------------+---------+------+------------+
|total_filas|nulos|no_disponible_102_3|invalidos|maximo|percentil_99|
+-----------+-----+-------------------+---------+------+------------+
|   60533559|    0|             159987|    26757| 102.3|        22.0|
+-----------+-----+-------------------+---------+------+------------+

+----------+----------------+-----+----------+
|VesselType|TransceiverClass|filas|sog_maximo|
+----------+----------------+-----+----------+
|        37|               B|10356|     102.2|
|        60|               A| 3564|      83.7|
|        90|               A| 3193|     102.2|
|        51|               A| 1993|      46.6|
|        40|               A| 1492|      74.6|
|        55|               A| 1047|      53.8|
|        65|               A|  851|      38.8|
|        51|               B|  675|      93.0|
|         0|               A|  586|     102.2|
|        30|               B|  541|     102.2|
+----------+----------------+-----+---

#### Ficha R3 · `SOG`

**Categoría.** Valor inválido por límite de dominio, con 102,3 contado aparte.

**Resultado.**

| Medida | Filas | % del total |
| --- | --- | --- |
| Nulos | 0 | 0,00% |
| Valor 102,3 (no disponible) | 159.987 | 0,2643% |
| Sobre el umbral de 35 nudos | 26.757 | 0,0442% |

El máximo de la columna es 102,3. El percentil 99 es 22,0 nudos.

**Lectura 1. El umbral de 35 nudos es holgado.** El percentil 99 vale 22,0 nudos, de modo que el 99% de las posiciones queda por debajo de esa velocidad. El umbral de 35 nudos deja fuera solo el 0,0442% de las filas. Un umbral más bajo marcaría tráfico rápido legítimo.

**Lectura 2. Los valores extremos llegan hasta 102,2.** Varios grupos alcanzan un máximo de 102,2 nudos, que es el tope del rango técnico del campo. El valor 102,3 queda reservado para no disponible. Un valor exacto de 102,2 indica saturación del campo y no una medición real.

**Lectura 3. Dónde se concentran las marcas.** Los diez grupos de la evidencia reúnen 24.298 filas, que es el 90,8% de las 26.757 marcadas.

| `VesselType` | Clase | Filas | % de las marcadas | `SOG` máximo |
| --- | --- | --- | --- | --- |
| 37 Pleasure craft | B | 10.356 | 38,7% | 102,2 |
| 60 Passenger | A | 3.564 | 13,3% | 83,7 |
| 90 Other type | A | 3.193 | 11,9% | 102,2 |
| 51 Search and rescue | A | 1.993 | 7,4% | 46,6 |
| 40 High speed craft | A | 1.492 | 5,6% | 74,6 |
| 55 Law enforcement | A | 1.047 | 3,9% | 53,8 |
| 65 Passenger | A | 851 | 3,2% | 38,8 |
| 51 Search and rescue | B | 675 | 2,5% | 93,0 |
| 0 Sin tipo reportado | A | 586 | 2,2% | 102,2 |
| 30 Fishing | B | 541 | 2,0% | 102,2 |

**Lectura 4. La regla marca tráfico rápido legítimo.** Los códigos 40 (high speed craft), 51 (search and rescue) y 55 (law enforcement) agrupan embarcaciones que superan 35 nudos en servicio normal. Esos tres códigos suman 5.207 filas, que es el 19,5% de las marcadas. El umbral fijo confunde ese tráfico con el error de sensor.

**Lectura 5. La clase B concentra el error de saturación.** El código 37 con transpondedor Clase B aporta 10.356 filas con máximo en 102,2. Las embarcaciones de recreo con equipo Clase B rara vez superan 40 nudos. El valor saturado indica un problema del equipo emisor.

**Decisión técnica 1. Origen del umbral de 35 nudos.** El umbral proviene del dominio naval. El estándar AIS define el rango técnico del campo, que llega hasta 102,2 nudos. Los buques mercantes de este dataset operan por debajo de 25 nudos. Las embarcaciones rápidas alcanzan 30 a 35 nudos.

**Decisión técnica 2. Umbral en una constante.** `SOG_MAX` queda expuesto como constante al inicio de la celda. Una revisión posterior ajusta el criterio en un solo punto del notebook.

**Decisión técnica 3. El valor 102,3 va separado.** El valor 102,3 corresponde al código de velocidad no disponible del protocolo. Ese conteo va en la columna `no_disponible_102_3`. La columna `invalidos` mide solo los valores que el emisor reporta como reales.

**Refinamiento registrado.** Una regla por tipo de buque asignaría un umbral distinto a los códigos 40, 51 y 55. Esa versión reduce el marcado de tráfico rápido legítimo. La mejora queda pendiente de prioridad media para la Entrega 2.

**Vínculo con 3c.** Las filas con `SOG` saturado señalan a los mismos emisores que producen saltos de distancia. El notebook 03 cruza esta lista con los buques de mayor distancia recorrida.

#### R4 · `Heading` con el valor 511 de no disponible

In [0]:
# R4. Heading viaja en 9 bits, con rango técnico de 0 a 359 grados.
# El estándar ITU-R M.1371 reserva 511 para "rumbo no disponible".
# El conteo de 511 va en columna propia, separado del conteo de valores fuera de rango.
ais.select(
    F.count("*").alias("total_filas"),
    F.sum(F.col("Heading").isNull().cast("long")).alias("nulos"),
    F.sum((F.col("Heading") == 511).cast("long")).alias("no_disponible_511"),
    F.sum(
        ((F.col("Heading") < 0) | ((F.col("Heading") > 360) & (F.col("Heading") != 511))).cast("long")
    ).alias("fuera_de_rango"),
).show()

# Evidencia: reparto del Flag por clase de transpondedor.
ais.groupBy("TransceiverClass").agg(
    F.count("*").alias("filas"),
    F.sum((F.col("Heading") == 511).cast("long")).alias("heading_511"),
    F.round(F.avg((F.col("Heading") == 511).cast("int")) * 100, 1).alias("pct_511"),
).display()

+-----------+-----+-----------------+--------------+
|total_filas|nulos|no_disponible_511|fuera_de_rango|
+-----------+-----+-----------------+--------------+
|   60533559|    0|         33535628|            19|
+-----------+-----+-----------------+--------------+



TransceiverClass,filas,heading_511,pct_511
A,40581162,16028300,39.5
B,19952397,17507328,87.7


#### Ficha R4 · `Heading`

**Categoría.** Valor de no disponible.

**Resultado.** La columna no tiene nulos. El valor 511 aparece en 33.535.628 filas, que es el 55,40% del total. Fuera de 511, la corrida marca 19 filas con valor fuera de rango, que es el 0,00003% del total.

**Resultado por clase de transpondedor.**

| Clase | Filas | Filas con 511 | % con 511 |
| --- | --- | --- | --- |
| A | 40.581.162 | 16.028.300 | 39,5% |
| B | 19.952.397 | 17.507.328 | 87,7% |

**Lectura.** La Clase B concentra la proporción más alta. Ese resultado es consistente con la ausencia de sensor de rumbo en el equipo de Clase B, que muchas embarcaciones menores usan sin girocompás conectado. La proporción de 39,5% en la Clase A queda como hallazgo abierto. Los datos disponibles no permiten asignarle una causa.

**Decisión técnica 1. El valor 511 ocupa una categoría propia.** El valor 511 indica que el equipo declara el rumbo como no disponible. El conteo va en la columna `no_disponible_511`. La tabla consolidada de 2.6 lo reporta bajo la categoría de valor de no disponible.

**Decisión técnica 2. Exclusión en los análisis de rumbo.** Cualquier cálculo que use `Heading` filtra primero las filas con 511. La cobertura del análisis baja entonces al 44,60% de las filas, y esa cobertura se declara junto al resultado.

**Precisión pendiente de prioridad baja.** La condición actual usa `Heading > 360`, por lo que el valor exacto 360 no queda marcado. El rango del estándar es 0 a 359. El ajuste del criterio mueve el conteo solo en la cantidad de filas con `Heading = 360`, sobre una base actual de 19 filas.

#### R5 · `IMO` ausente, con `IMO0000000` o con formato irregular

In [0]:
# R5. El número IMO identifica el casco de forma permanente, con 7 dígitos.
# La convención SOLAS lo exige solo a buques de carga y pasaje sobre cierto tonelaje.
# El valor IMO0000000 es el relleno de la fuente para los buques sin número asignado.
# Nota: IMO0000000 cumple el patrón de 7 dígitos, por lo que se resta del conteo válido.
IMO_FORMATO = F.col("IMO").rlike(r"^IMO\d{7}$")
IMO_NO_DISPONIBLE = F.col("IMO") == "IMO0000000"

IMO_AUSENTE = F.col("IMO").isNull() | IMO_NO_DISPONIBLE
IMO_IRREGULAR = ~IMO_FORMATO & F.col("IMO").isNotNull()

ais.select(
    F.count("*").alias("total_filas"),
    F.sum(F.col("IMO").isNull().cast("long")).alias("nulos"),
    F.sum(IMO_NO_DISPONIBLE.cast("long")).alias("no_disponible_IMO0000000"),
    F.sum(IMO_FORMATO.cast("long")).alias("formato_valido"),
    F.sum(IMO_IRREGULAR.cast("long")).alias("otros_formatos"),
).show()

# Evidencia: valores que no cumplen el patrón IMO + 7 dígitos.
ais.where(IMO_IRREGULAR).select("MMSI", "IMO", "VesselName").show(20, truncate=False)

+-----------+--------+------------------------+--------------+--------------+
|total_filas|   nulos|no_disponible_IMO0000000|formato_valido|otros_formatos|
+-----------+--------+------------------------+--------------+--------------+
|   60533559|25891448|                13733482|      34241949|        400162|
+-----------+--------+------------------------+--------------+--------------+

+---------+-------------+-----------------+
|MMSI     |IMO          |VesselName       |
+---------+-------------+-----------------+
|368052580|IMO101233527 |WOOD RIVER       |
|338070397|IMO600000000 |ACOUSTIC EXPLORER|
|368227820|IMO180000000 |MISS WRENN       |
|368159040|IMO1050000000|VELMA C          |
|367645040|IMO939738900 |GASPARILLA       |
|368063040|IMO101074535 |HASKELL          |
|368139280|IMO896876500 |MACKENZIE ROSE   |
|368145640|IMO826000000 |MARC             |
|368139280|IMO896876500 |MACKENZIE ROSE   |
|367730160|IMO101259957 |MANTEO           |
|367730160|IMO101259957 |MANTEO      

#### Ficha R5 · `IMO`

**Categoría.** Valor de no disponible (ausencia) y valor inválido de formato, contados por separado.

**Resultado.**

| Situación | Filas | % del total |
| --- | --- | --- |
| `IMO` nulo | 25.891.448 | 42,77% |
| `IMO0000000` | 13.733.482 | 22,69% |
| IMO asignado con formato correcto | 20.508.467 | 33,88% |
| Formato irregular | 400.162 | 0,66% |

La columna `formato_valido` de la corrida entrega 34.241.949 filas. Ese total incluye `IMO0000000`, porque `IMO0000000` cumple el patrón de 7 dígitos. La resta entrega las 20.508.467 filas con IMO asignado real.

**Lectura de la evidencia.** Los valores irregulares traen 9 o 10 dígitos: "IMO101233527", "IMO1050000000", "IMO939738900". El estándar usa 7 dígitos. Los nombres asociados ("WOOD RIVER", "MANTEO", "GASPARILLA", "MACKENZIE ROSE") corresponden a remolcadores y embarcaciones de servicio de la costa de Estados Unidos.

**Decisión técnica 1. Ausencia y composición de flota.** El nulo y el valor `IMO0000000` describen la misma situación: el buque opera sin número IMO asignado. Las dos categorías se agrupan como "sin IMO", con 39.624.930 filas (65,46%). Esa cifra describe la composición de la flota bajo la convención SOLAS. La Entrega 2 la reporta como característica del dataset.

**Decisión técnica 2. El formato irregular sí es un problema de calidad.** Las 400.162 filas con 9 o 10 dígitos representan un error de transcripción o de decodificación en el origen. Ese grupo entra en la tabla consolidada como regla de calidad con 0,66% de las filas.

**Decisión técnica 3. `IMO` queda fuera de las llaves de identidad.** La cobertura útil de la columna es 33,88%. Los análisis de identidad de buque usan `MMSI` como llave. `IMO` entra solo en verificaciones cruzadas puntuales.

**Mejora opcional registrada.** El séptimo dígito del IMO es un dígito de control. La suma ponderada de los primeros seis dígitos, con pesos 7, 6, 5, 4, 3 y 2, termina en el valor del dígito de control. Esa verificación separa los identificadores mal transcritos de los identificadores correctos. La celda queda pendiente de prioridad baja.

#### R6 · Filas con `_corrupt_record`

In [0]:
# R6. La lectura de 1.4 usa modo PERMISSIVE con columnNameOfCorruptRecord.
# Spark guarda en esa columna la línea completa que no calza con el esquema.
# Un valor nulo en la columna indica que la fila calzó sin problema.
FILA_CORRUPTA = F.col("_corrupt_record").isNotNull()

ais.select(
    F.count("*").alias("total_filas"),
    F.sum(F.col("_corrupt_record").isNull().cast("long")).alias("nulos"),
    F.sum(FILA_CORRUPTA.cast("long")).alias("corruptas"),
).show()

# Evidencia: la tabla debe salir vacía cuando el conteo es cero.
ais.where(FILA_CORRUPTA).select("_corrupt_record").show(20, truncate=False)

+-----------+--------+---------+
|total_filas|   nulos|corruptas|
+-----------+--------+---------+
|   60533559|60533559|        0|
+-----------+--------+---------+

+---------------+
|_corrupt_record|
+---------------+
+---------------+



#### Ficha R6 · `_corrupt_record`

**Categoría.** Integridad de lectura.

**Resultado.** 0 filas corruptas. Las 60.533.559 filas traen `_corrupt_record` nulo.

**Decisión técnica 1. Qué demuestra el resultado.** El esquema explícito de 1.4 coincide con la estructura de los 7 archivos CSV. Las 18 columnas declaradas, sus tipos y el formato de fecha aceptan todas las líneas del origen. El requisito 1 del enunciado queda respaldado con esta evidencia.

**Decisión técnica 2. Conservar la regla.** La regla permanece en el notebook como control de regresión. Una carga futura con archivos de otro periodo puede traer un cambio de estructura en el origen, y esta celda lo detecta en la primera corrida.

**Alcance de la regla.** `_corrupt_record` detecta fallas de estructura, por ejemplo una columna de más o una comilla sin cerrar. Las fallas de contenido, por ejemplo una velocidad imposible, quedan a cargo de las reglas R1 a R5.

#### R7 · Filas fuera del día del archivo de origen

In [0]:
# R7. La versión anterior comparaba to_date(BaseDateTime)
# contra la columna day. La ingesta de 1.4 deriva day con F.to_date("BaseDateTime"),
# por lo que esa comparación nunca puede fallar y devolvía cero por construcción.
# La verificación real toma la fecha del nombre del archivo de origen.
FECHA_DEL_ARCHIVO = F.to_date(
    F.regexp_extract(F.col("source_file"), r"AIS_(\d{4}_\d{2}_\d{2})", 1),
    "yyyy_MM_dd",
)

FUERA_DEL_DIA = (
    FECHA_DEL_ARCHIVO.isNull()
    | F.col("BaseDateTime").isNull()
    | (F.to_date("BaseDateTime") != FECHA_DEL_ARCHIVO)
)

ais.select(
    F.count("*").alias("total_filas"),
    F.sum((~FUERA_DEL_DIA).cast("long")).alias("coinciden"),
    F.sum(FUERA_DEL_DIA.cast("long")).alias("no_coinciden"),
).show()

# Evidencia: filas cuya fecha no coincide con la fecha del archivo que las trajo.
(
    ais.where(FUERA_DEL_DIA)
    .select("MMSI", "BaseDateTime", "day", "source_file")
    .show(20, truncate=False)
)

+-----------+---------+------------+
|total_filas|coinciden|no_coinciden|
+-----------+---------+------------+
|   60533559| 60533559|           0|
+-----------+---------+------------+

+----+------------+---+-----------+
|MMSI|BaseDateTime|day|source_file|
+----+------------+---+-----------+
+----+------------+---+-----------+



#### Ficha R7 · Coherencia entre fila y archivo

**Categoría.** Integridad de la partición temporal.

**Resultado.** 60.533.559 filas coinciden. 0 filas quedan fuera del día de su archivo de origen.

**Corrección aplicada.** La celda anterior comparaba `to_date(BaseDateTime)` contra la columna `day`. La ingesta de 1.4 crea `day` con la expresión `F.to_date("BaseDateTime")`. La comparación enfrentaba una columna contra su propia derivación, y el resultado de 0 filas era cierto por construcción. Esa corrida no aportaba evidencia sobre el origen de los datos.

**Criterio actual.** La fecha de referencia sale del nombre del archivo con `regexp_extract` sobre `source_file`, por ejemplo `AIS_2023_06_01.csv`. La comparación enfrenta ahora dos fuentes independientes: el timestamp que transmite el buque y el corte por día que aplica MarineCadastre.

**Lectura.** El resultado en cero con el criterio corregido demuestra tres cosas. El origen corta los archivos por día en UTC, igual que `BaseDateTime`. La sesión fija `spark.sql.session.timeZone` en UTC en 1.0, por lo que la conversión no desplaza el día. Ninguna fila con reloj desviado llegó a un archivo de otro día.

**Decisión técnica 1. `day` es una columna derivada segura.** La columna `day` reproduce el corte del origen, por lo que sirve como llave de partición sin recálculo. Esa equivalencia queda demostrada con evidencia y no por supuesto.

**Decisión técnica 2. Insumo de la sección 4.** Una partición por `day` produce archivos alineados con los archivos de origen. Una consulta que filtra por fecha descarta 6 de las 7 particiones sin abrirlas.

#### R8 · Duplicados exactos de fila completa

In [0]:
# R8. Una fila duplicada exacta repite el valor de todas las columnas de negocio.
# Las columnas de comparación excluyen _corrupt_record e ingested_at.
# ingested_at toma un valor constante en esta carga, por lo que su exclusión
# no altera el conteo.
business_cols = [c for c in ais.columns if c not in ("_corrupt_record", "ingested_at")]

duplicados_exactos = (
    ais.groupBy(business_cols)
    .agg(F.count("*").alias("veces"))
    .where("veces > 1")
)

total_duplicados = duplicados_exactos.agg(
    F.sum("veces").alias("total_filas_duplicadas"),
    F.count("*").alias("grupos_duplicados"),
    F.max("veces").alias("max_repeticiones"),
).collect()[0]

filas_dup_exactas = int(total_duplicados["total_filas_duplicadas"] or 0)

print(f"Total de filas que participan en duplicados exactos: {total_duplicados['total_filas_duplicadas']:,}")
print(f"Número de grupos de filas duplicadas: {total_duplicados['grupos_duplicados']:,}")
print(f"Máximo de repeticiones de una fila: {total_duplicados['max_repeticiones']:,}")

print("\nEjemplos de filas duplicadas (top 10 grupos):")
duplicados_exactos.orderBy(F.desc("veces")).show(10, truncate=False)

Total de filas que participan en duplicados exactos: 2,776
Número de grupos de filas duplicadas: 1,388
Máximo de repeticiones de una fila: 2

Ejemplos de filas duplicadas (top 10 grupos):
+---------+-------------------+--------+----------+----+-----+-------+------------------+----------+--------+----------+------+------+-----+-----+-----+----------------+------------------+----------+-----+
|MMSI     |BaseDateTime       |LAT     |LON       |SOG |COG  |Heading|VesselName        |IMO       |CallSign|VesselType|Status|Length|Width|Draft|Cargo|TransceiverClass|source_file       |day       |veces|
+---------+-------------------+--------+----------+----+-----+-------+------------------+----------+--------+----------+------+------+-----+-----+-----+----------------+------------------+----------+-----+
|319139200|2023-06-02 01:00:00|30.9861 |-117.29891|10.4|356.9|356.0  |SHERPA            |IMO9795529|ZGHE9   |37        |0     |74.0  |13.0 |3.7  |37   |A               |AIS_2023_06_02.csv|2023-0

#### Ficha R8 · Duplicados exactos

**Categoría.** Integridad del pipeline de ingesta.

**Resultado.** 2.776 filas participan en un duplicado exacto, que es el 0,0046% del total. Los grupos son 1.388. El máximo de repeticiones de una misma fila es 2.

**Lectura de la evidencia.** Las marcas de tiempo de los ejemplos caen en el borde de la hora: "21:00:00", "17:59:59", "22:59:59", "11:59:59". El patrón apunta a un solape en el corte por hora del proceso de agregación de la fuente. Los ejemplos cubren varios días y varias clases de transpondedor, por lo que el efecto no se concentra en un archivo.

**Decisión técnica 1. Columnas de comparación.** La comparación excluye `_corrupt_record` e `ingested_at`. `ingested_at` viene de `current_timestamp()` y toma el mismo valor en toda la carga, de modo que su exclusión deja el conteo igual. `source_file` y `day` permanecen en la comparación, porque describen el origen real de la fila.

**Decisión técnica 2. Qué descarta este resultado.** Una doble inserción de archivos en la ingesta produce una proporción alta de duplicados exactos. La proporción medida es 0,0046%, por lo que el pipeline de 1.4 cargó cada archivo una sola vez. La reconciliación de 1.6 respalda la misma conclusión.

**Decisión técnica 3. Tratamiento.** Las filas se conservan en `ais_raw`. La tabla cruda mantiene el dato tal como llega del origen. La deduplicación se decide en la Entrega 2, junto con el resto de las reglas de limpieza.

#### R9 · Duplicados por la llave (`MMSI`, `BaseDateTime`)

In [0]:
# R9. Un buque no puede ocupar dos posiciones en el mismo instante.
# La llave (MMSI, BaseDateTime) debe identificar una sola fila.
# El conteo de posiciones distintas separa la repetición idéntica de la
# repetición con coordenadas diferentes.
duplicados_llave = (
    ais.groupBy("MMSI", "BaseDateTime")
    .agg(
        F.count("*").alias("veces"),
        F.countDistinct("LAT", "LON").alias("posiciones_distintas"),
    )
    .where("veces > 1")
)

resumen_llave = duplicados_llave.agg(
    F.count("*").alias("grupos_duplicados"),
    F.sum("veces").alias("total_filas_duplicadas"),
    F.sum((F.col("posiciones_distintas") > 1).cast("long")).alias("grupos_con_posiciones_distintas"),
    F.max("veces").alias("max_repeticiones"),
).collect()[0]

filas_dup_llave = int(resumen_llave["total_filas_duplicadas"] or 0)

print(f"Grupos con (MMSI, BaseDateTime) repetido: {resumen_llave['grupos_duplicados']:,}")
print(f"Total de filas que participan en duplicados por llave: {resumen_llave['total_filas_duplicadas']:,}")
print(f"Grupos donde las posiciones (LAT, LON) difieren: {resumen_llave['grupos_con_posiciones_distintas']:,}")
print(f"Máximo de repeticiones de una combinación: {resumen_llave['max_repeticiones']:,}")

print("\nEjemplos de duplicados por llave con posiciones distintas (top 10):")
duplicados_llave.where("posiciones_distintas > 1").orderBy(F.desc("veces")).show(10, truncate=False)

Grupos con (MMSI, BaseDateTime) repetido: 1,672
Total de filas que participan en duplicados por llave: 3,344
Grupos donde las posiciones (LAT, LON) difieren: 272
Máximo de repeticiones de una combinación: 2

Ejemplos de duplicados por llave con posiciones distintas (top 10):
+---------+-------------------+-----+--------------------+
|MMSI     |BaseDateTime       |veces|posiciones_distintas|
+---------+-------------------+-----+--------------------+
|303273000|2023-06-04 20:26:09|2    |2                   |
|538005797|2023-06-04 19:34:08|2    |2                   |
|338428281|2023-06-04 22:54:32|2    |2                   |
|538005797|2023-06-04 17:20:08|2    |2                   |
|249430000|2023-06-04 16:56:58|2    |2                   |
|367401030|2023-06-04 18:54:19|2    |2                   |
|303273000|2023-06-04 18:01:19|2    |2                   |
|303273000|2023-06-04 16:58:29|2    |2                   |
|367409000|2023-06-04 20:05:18|2    |2                   |
|538005797|2023-

#### Ficha R9 · Duplicados por llave

**Categoría.** Violación de unicidad lógica.

**Resultado.** 1.672 combinaciones de (`MMSI`, `BaseDateTime`) se repiten. 3.344 filas participan, que es el 0,0055% del total. En 272 grupos las coordenadas difieren. El máximo de repeticiones de una combinación es 2.

**Relación con R8.** Los duplicados exactos suman 1.388 grupos. Los duplicados por llave suman 1.672 grupos. La diferencia de 284 grupos corresponde a filas que comparten la llave y difieren en alguna columna. De esos, 272 grupos difieren en `LAT` o `LON`.

**Causa técnica probable.** La red terrestre de AIS opera con varias estaciones receptoras. Una misma transmisión de radio puede llegar a dos estaciones. El proceso de recolección entrega entonces dos filas con la misma llave y una diferencia mínima en las coordenadas decodificadas.

**Límite de la evidencia mostrada.** Los diez ejemplos comparten el valor 2 en la columna `veces`. El orden descendente no distingue entre grupos empatados, por lo que la muestra no indica concentración por día. Un conteo de grupos por día confirma o descarta esa concentración, y queda registrado como celda opcional.

**Decisión técnica 1. Prioridad para la pregunta 3c.** El cálculo de distancia usa `lag` sobre una ventana particionada por `MMSI` y ordenada por `BaseDateTime`. Dos filas con la misma llave y coordenadas distintas producen un desplazamiento con intervalo de tiempo cero. En este case **se debe** hacer una deduplicacion.

**Decisión técnica 2. Magnitud acotada.** La proporción es 0,0055%. El efecto sobre los agregados de distancia es bajo. El efecto sobre los valores extremos es alto, porque cada grupo produce un salto artificial que domina el máximo.

### 2.5 Columnas complementarias


In [0]:
# COG es el rumbo sobre el fondo, en grados.
# El estándar reserva 360,0 para "rumbo sobre el fondo no disponible".
# Es el tercer valor de este tipo en el dataset, junto con Heading = 511 y SOG = 102,3.
ais.select(
    F.count("*").alias("total"),
    F.sum(F.col("COG").isNull().cast("long")).alias("nulos"),
    F.sum((F.col("COG") == 360.0).cast("long")).alias("no_disponible_360"),
    F.sum(((F.col("COG") < 0) | (F.col("COG") > 360)).cast("long")).alias("fuera_de_rango"),
).show()

+--------+-----+-----------------+--------------+
|   total|nulos|no_disponible_360|fuera_de_rango|
+--------+-----+-----------------+--------------+
|60533559|    0|         10291131|            12|
+--------+-----+-----------------+--------------+



#### Ficha 2.5a · `COG`

**Categoría.** Valor de no disponible.

**Resultado.** La columna no tiene nulos. 10.291.131 filas traen el valor 360,0, que es el 17,00% del total. 12 filas quedan fuera del rango 0 a 360.

**Patrón confirmado.** El dataset presenta tres valores de no disponible con la misma mecánica: `Heading = 511` en 9 bits, `SOG = 102.3` en 10 bits y `COG = 360.0`. En los tres casos el protocolo usa el valor máximo del campo para declarar el dato como no disponible. Esta convención explica por qué un perfil ingenuo marca esos valores como atípicos.

**Comparación con `Heading`.** `COG` reporta dato no disponible en el 17,00% de las filas. `Heading` lo reporta en el 55,40%. La diferencia tiene sentido técnico: `COG` se calcula a partir de posiciones GPS sucesivas, y `Heading` requiere un girocompás instalado a bordo.

**Decisión técnica.** Los análisis de rumbo filtran el valor 360,0 antes de agregar. La cobertura del análisis con `COG` es 83,00% de las filas, frente a 44,60% con `Heading`. Los cálculos de dirección usan `COG` como fuente principal por esa razón.

In [0]:
# Completitud de las columnas descriptivas y operativas.
resto_cols = ["VesselName", "CallSign", "Status", "Draft", "Cargo"]

ais.select(
    F.count("*").alias("total"),
    *[F.sum(F.col(c).isNull().cast("long")).alias(f"nulos_{c}") for c in resto_cols],
).show()

+--------+----------------+--------------+------------+-----------+-----------+
|   total|nulos_VesselName|nulos_CallSign|nulos_Status|nulos_Draft|nulos_Cargo|
+--------+----------------+--------------+------------+-----------+-----------+
|60533559|          150685|      10157915|    19952397|   39000865|   19898489|
+--------+----------------+--------------+------------+-----------+-----------+



In [0]:
# Tabla cruzada de nulos por clase de transpondedor.
# Confirma si la ausencia de Status, Draft y Cargo depende del tipo de mensaje AIS.
ais.groupBy("TransceiverClass").agg(
    F.count("*").alias("filas"),
    F.round(F.avg(F.col("Status").isNull().cast("int")) * 100, 1).alias("pct_status_nulo"),
    F.round(F.avg(F.col("Draft").isNull().cast("int")) * 100, 1).alias("pct_draft_nulo"),
    F.round(F.avg(F.col("Cargo").isNull().cast("int")) * 100, 1).alias("pct_cargo_nulo"),
    F.round(F.avg(F.col("CallSign").isNull().cast("int")) * 100, 1).alias("pct_callsign_nulo"),
).display()

TransceiverClass,filas,pct_status_nulo,pct_draft_nulo,pct_cargo_nulo,pct_callsign_nulo
A,40581162,0.0,47.5,0.0,4.0
B,19952397,100.0,98.8,99.7,42.8


#### Ficha 2.5b · Completitud

**Categoría.** Valor ausente.

**Resultado sobre el total.**

| Columna | Filas nulas | % del total |
| --- | --- | --- |
| `VesselName` | 150.685 | 0,25% |
| `CallSign` | 10.157.915 | 16,78% |
| `Status` | 19.952.397 | 32,96% |
| `Cargo` | 19.898.489 | 32,87% |
| `Draft` | 39.000.865 | 64,43% |

**Resultado por clase de transpondedor.**

| Clase | Filas | `Status` nulo | `Draft` nulo | `Cargo` nulo | `CallSign` nulo |
| --- | --- | --- | --- | --- | --- |
| A | 40.581.162 | 0,0% | 47,5% | 0,0% | 4,0% |
| B | 19.952.397 | 100,0% | 98,8% | 99,7% | 42,8% |

**Lectura 1. `Status` y `Cargo` dependen por completo de la clase.** La Clase A reporta las dos columnas en el 100% de sus filas. La Clase B las deja nulas en el 100% y el 99,7%. Los mensajes de Clase A (tipos 1, 2, 3 para posición y tipo 5 para datos estáticos) transportan esos campos. Los mensajes de Clase B (tipos 18 y 24) los omiten en su estructura. La ausencia proviene del diseño del protocolo.

**Lectura 2. `Draft` tiene un vacío real dentro de la Clase A.** El 47,5% de las filas de Clase A llega sin calado. El calado es un dato que la tripulación introduce a mano y actualiza en cada viaje. Este es el único caso del grupo donde la ausencia refleja una práctica operativa y no una limitación del mensaje.

**Lectura 3. `CallSign` falta en el 42,8% de la Clase B.** El indicativo de llamada se asigna con la licencia de radio. Muchas embarcaciones menores operan sin esa licencia.

**Decisión técnica 1. Límite estructural documentado.** Estas ausencias quedan registradas como característica del protocolo. La Entrega 2 las reporta sin una regla de limpieza asociada.

**Decisión técnica 2. Cobertura declarada.** Todo análisis que use `Status` o `Cargo` cubre solo la flota de Clase A, que es el 67,04% de las filas. Esa cobertura se declara junto al resultado.

**Decisión técnica 3. Una sola pasada con `avg` sobre un booleano.** La tabla cruzada usa `F.avg(columna.isNull().cast("int"))` para obtener la proporción directa. Ese patrón evita dividir un `sum` entre un `count` por cada columna y deja las cuatro proporciones en un solo `groupBy`.

**Vínculo con R3.** `Status` distingue un buque fondeado de un buque en navegación. El valor está disponible en el 100% de la Clase A. Un umbral de `SOG` por estado de navegación es viable para esa parte de la flota.

### 2.6 Diagnóstico consolidado

La siguiente celda aplica las 7 reglas de fila en una sola pasada sobre la tabla. Las dos reglas de duplicados necesitan agregación por grupo, por lo que reutilizan los valores calculados en R8 y R9.

In [0]:
# Requiere que R1 a R9 se ejecuten antes, en orden,
# porque reutiliza las condiciones y los conteos definidos en esas celdas.

REGLAS_DE_FILA = {
    "R1_mmsi_formato_invalido":      ~MMSI_VALIDO,
    "R2_coordenadas_fuera_de_rango": COORD_FUERA,
    "R3_sog_sobre_umbral":           SOG_INVALIDO,
    "R4_heading_no_disponible_511":  F.col("Heading") == 511,
    "R5a_imo_ausente":               IMO_AUSENTE,
    "R5b_imo_formato_irregular":     IMO_IRREGULAR,
    "R6_fila_corrupta":              FILA_CORRUPTA,
    "R7_fuera_del_dia_del_archivo":  FUERA_DEL_DIA,
}

CATEGORIA = {
    "R1_mmsi_formato_invalido":      "Valor invalido",
    "R2_coordenadas_fuera_de_rango": "Valor invalido",
    "R3_sog_sobre_umbral":           "Valor invalido",
    "R4_heading_no_disponible_511":  "Valor no disponible",
    "R5a_imo_ausente":               "Valor no disponible",
    "R5b_imo_formato_irregular":     "Valor invalido",
    "R6_fila_corrupta":              "Integridad de lectura",
    "R7_fuera_del_dia_del_archivo":  "Integridad temporal",
    "R8_duplicado_exacto":           "Duplicado",
    "R9_duplicado_por_llave":        "Duplicado",
}

conteos = ais.agg(
    *[F.sum(cond.cast("long")).alias(nombre) for nombre, cond in REGLAS_DE_FILA.items()]
).collect()[0].asDict()

conteos["R8_duplicado_exacto"] = filas_dup_exactas
conteos["R9_duplicado_por_llave"] = filas_dup_llave

filas = [
    (nombre, CATEGORIA[nombre], int(valor or 0), round(100.0 * (valor or 0) / TOTAL_FILAS, 4))
    for nombre, valor in conteos.items()
]

resumen_calidad = spark.createDataFrame(
    filas, "regla string, categoria string, filas_afectadas long, pct_filas double"
).orderBy(F.desc("filas_afectadas"))

resumen_calidad.display()

regla,categoria,filas_afectadas,pct_filas
R5a_imo_ausente,Valor no disponible,39624930,65.4594
R4_heading_no_disponible_511,Valor no disponible,33535628,55.4001
R5b_imo_formato_irregular,Valor invalido,400162,0.6611
R1_mmsi_formato_invalido,Valor invalido,49897,0.0824
R3_sog_sobre_umbral,Valor invalido,26757,0.0442
R9_duplicado_por_llave,Duplicado,3344,0.0055
R8_duplicado_exacto,Duplicado,2776,0.0046
R2_coordenadas_fuera_de_rango,Valor invalido,0,0.0
R6_fila_corrupta,Integridad de lectura,0,0.0
R7_fuera_del_dia_del_archivo,Integridad temporal,0,0.0


In [0]:
# Esta celda persiste los resultados del perfilamiento de calidad en Delta
(
    resumen_calidad.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("ocean_watch.raw.ais_quality_report")
)

(
    volumen_diario.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("ocean_watch.raw.ais_daily_profile")
)

COMENTARIO_CALIDAD = (
    "Diagnostico de calidad de ocean_watch.raw.ais. Una fila por regla R1 a R9, "
    "con filas afectadas y porcentaje sobre el total. Insumo de la Entrega 2."
)
COMENTARIO_PERFIL = (
    "Perfil de volumen diario de ocean_watch.raw.ais: posiciones, buques exactos "
    "y buques aproximados por dia."
)

spark.sql(f"COMMENT ON TABLE ocean_watch.raw.ais_quality_report IS '{COMENTARIO_CALIDAD}'")
spark.sql(f"COMMENT ON TABLE ocean_watch.raw.ais_daily_profile IS '{COMENTARIO_PERFIL}'")

display(spark.table("ocean_watch.raw.ais_quality_report"))

regla,categoria,filas_afectadas,pct_filas
R5a_imo_ausente,Valor no disponible,39624930,65.4594
R4_heading_no_disponible_511,Valor no disponible,33535628,55.4001
R5b_imo_formato_irregular,Valor invalido,400162,0.6611
R1_mmsi_formato_invalido,Valor invalido,49897,0.0824
R3_sog_sobre_umbral,Valor invalido,26757,0.0442
R9_duplicado_por_llave,Duplicado,3344,0.0055
R8_duplicado_exacto,Duplicado,2776,0.0046
R2_coordenadas_fuera_de_rango,Valor invalido,0,0.0
R6_fila_corrupta,Integridad de lectura,0,0.0
R7_fuera_del_dia_del_archivo,Integridad temporal,0,0.0


#### Ficha 2.6 · Diagnóstico consolidado

**Resultado.** Las 10 reglas sobre 60.533.559 filas.

| Regla | Categoría | Filas afectadas | % del total | Lectura |
| --- | --- | --- | --- | --- |
| R5a `IMO` ausente | Valor no disponible | 39.624.930 | 65,4594% | Flota sin obligación de número IMO bajo SOLAS |
| R4 `Heading` 511 | Valor no disponible | 33.535.628 | 55,4001% | Equipo sin sensor de rumbo, sobre todo Clase B |
| R5b `IMO` formato irregular | Valor inválido | 400.162 | 0,6611% | Error de transcripción en el origen |
| R1 `MMSI` formato inválido | Valor inválido | 49.897 | 0,0824% | Ayudas a la navegación y buques de gobierno |
| R3 `SOG` sobre el umbral | Valor inválido | 26.757 | 0,0442% | Saturación del campo y tráfico rápido legítimo |
| R9 Duplicado por llave | Duplicado | 3.344 | 0,0055% | Doble captación por estaciones receptoras |
| R8 Duplicado exacto | Duplicado | 2.776 | 0,0046% | Solape en el corte por hora del origen |
| R2 Coordenadas fuera de rango | Valor inválido | 0 | 0,0000% | Rango geográfico íntegro |
| R6 Fila corrupta | Integridad de lectura | 0 | 0,0000% | El esquema explícito acepta los 7 archivos |
| R7 Fuera del día del archivo | Integridad temporal | 0 | 0,0000% | El corte por día del origen coincide con UTC |

**Conclusión 1. El dataset llega estructuralmente íntegro.** Las tres reglas de estructura (R2, R6, R7) dan cero. Los duplicados (R8, R9) suman 6.120 filas, que es el 0,0101% del total. El esquema explícito de 1.4, la verificación de integridad de 1.2 y la reconciliación de 1.6 explican ese resultado.

**Conclusión 2. El error de contenido es pequeño y localizado.** R1, R3 y R5b suman 476.816 filas como cota superior, que es el 0,7876% del total. Los tres grupos afectan identificadores y velocidad. Ninguno afecta la posición, que es la columna central del caso de uso.

**Conclusión 3. El volumen de hallazgos está en los valores de no disponible.** R4 cubre el 55,40% y R5a el 65,46%. Las dos cifras miden la cobertura real de cada columna dentro de la flota. Un análisis de rumbo trabaja sobre el 44,60% de las filas. Un análisis con llave IMO trabaja sobre el 33,88%.

**Conclusión 4. La flota se divide en dos poblaciones con calidad distinta.** La Clase A aporta el 67,04% de las filas, reporta `Status` y `Cargo` en el 100% de los casos y tiene 39,5% de rumbo no disponible. La Clase B aporta el 32,96%, deja `Status` y `Cargo` vacíos y tiene 87,7% de rumbo no disponible. El corte por `TransceiverClass` explica la mayor parte de los vacíos del dataset.

**Conclusión 5. El dato de posición es el más confiable.** `LAT` y `LON` tienen 0 nulos, 0 valores fuera de rango y 0 filas corruptas. Las preguntas de negocio que dependen de la posición (3c y 3d) operan sobre datos completos. Las preguntas que dependen de identidad (3a, 3e) descuentan el 0,08% de R1.

**Decisión técnica 1. Una sola pasada para las reglas de fila.** Las 8 reglas de fila viajan en un único `agg`. Spark resuelve el plan con un escaneo de la tabla. Una consulta por regla cuesta 8 escaneos de 60,5 millones de filas y entrega la misma información.

**Decisión técnica 2. Las reglas de duplicado necesitan intercambio de datos.** R8 y R9 agrupan por muchas columnas, por lo que Spark reparte los datos entre ejecutores antes de contar. Esas dos reglas quedan fuera del `agg` único y reutilizan el conteo que ya calcularon R8 y R9.

**Decisión técnica 3. Categoría junto al conteo.** Cada regla declara su categoría en la tabla persistida. Un lector de la Entrega 2 distingue un dato no disponible de un dato erróneo sin volver al notebook.

**Decisión técnica 4. Evidencia persistida en Delta.** `ocean_watch.raw.ais_quality_report` y `ocean_watch.raw.ais_daily_profile` quedan escritas con comentario de tabla. El resto del equipo lee esas tablas y evita repetir el cálculo sobre 60,5 millones de filas.

#### Pendientes registrados

| Pendiente | Prioridad | Motivo |
| --- | --- | --- |
| Umbral de `SOG` por tipo de buque | Media | Los códigos 40, 51 y 55 superan 35 nudos en servicio normal |
| Regla de rango por área de interés | Media | La caja envolvente llega a 89,66 de latitud, fuera de aguas de Estados Unidos |
| Dígito de control del `IMO` | Baja | Separa el error de transcripción del IMO correcto |
| Prefijo MID del `MMSI` | Baja | Verifica el país de matrícula contra la zona del dataset |
| Consistencia de dimensiones por `MMSI` | Baja | Mide el efecto de `dropDuplicates` en el perfil de flota |

# 3. Preguntas de negocio

Cada pregunta sigue la misma estructura: consulta, plan de ejecución con `explain("formatted")` y una ficha con el resultado, la lectura del plan y la decisión técnica. Las preguntas se responden sobre `ocean_watch.raw.ais` completa (60.533.559 filas). Solo 3c aplica filtros de calidad, porque la distancia acumulada es sensible a posiciones erróneas.

En serverless, Photon reemplaza parte de los operadores y el plan puede mostrar nombres como `PhotonGroupingAgg` o `PhotonShuffleExchangeSink` en lugar de `HashAggregate` o `Exchange`. Las fichas leen la forma del plan (número de intercambios, tipo de join, ejecución en Python), que no cambia con Photon.

### 3a. Buques distintos por día (`countDistinct` vs `approx_count_distinct`)

In [0]:
ais = spark.table("ocean_watch.raw.ais")

count = (
    ais
    .agg(F.count_distinct("MMSI").alias("count"))
    .collect()[0]
    .asDict()["count"]
)
print(f"Total de MMSI: {count:,}")

Total de MMSI: 31,871


In [0]:
ais = spark.table("ocean_watch.raw.ais")

distinct_exact = (
    ais
    .groupBy("day")
    .agg(F.count_distinct("MMSI").alias("count_mmsis"))
    .orderBy("day")
)

display(distinct_exact)

day,count_mmsis
2023-06-01,20448
2023-06-02,21153
2023-06-03,20505
2023-06-04,19720
2023-06-05,19615
2023-06-06,19717
2023-06-07,20086


In [0]:
ais = spark.table("ocean_watch.raw.ais")

distinct_approx = (
    ais
    .groupBy("day")
    .agg(F.approx_count_distinct("MMSI").alias("count_mmsis"))
    .orderBy("day")
)

display(distinct_approx)

day,count_mmsis
2023-06-01,21782
2023-06-02,23182
2023-06-03,20877
2023-06-04,20275
2023-06-05,20358
2023-06-06,20288
2023-06-07,21184


In [0]:
comparacion_3a = (
    distinct_exact.withColumnRenamed("count_mmsis", "exacto")
    .join(distinct_approx.withColumnRenamed("count_mmsis", "aproximado"), "day")
    .withColumn("error_pct", F.round((F.col("aproximado") - F.col("exacto")) / F.col("exacto") * 100, 2))
    .orderBy("day")
)
display(comparacion_3a)

day,exacto,aproximado,error_pct
2023-06-01,20448,21782,6.52
2023-06-02,21153,23182,9.59
2023-06-03,20505,20877,1.81
2023-06-04,19720,20275,2.81
2023-06-05,19615,20358,3.79
2023-06-06,19717,20288,2.9
2023-06-07,20086,21184,5.47


In [0]:
print("== count_distinct ==")
distinct_exact.explain("formatted")
print("== approx_count_distinct ==")
distinct_approx.explain("formatted")

== count_distinct ==
== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Initial Plan ==
   PhotonResultStage (17)
   +- PhotonColumnarToRow (16)
      +- PhotonSort (15)
         +- PhotonShuffleExchangeSource (14)
            +- PhotonShuffleMapStage (13)
               +- PhotonShuffleExchangeSink (12)
                  +- PhotonGroupingAgg (11)
                     +- PhotonShuffleExchangeSource (10)
                        +- PhotonShuffleMapStage (9)
                           +- PhotonShuffleExchangeSink (8)
                              +- PhotonGroupingAgg (7)
                                 +- PhotonGroupingAgg (6)
                                    +- PhotonShuffleExchangeSource (5)
                                       +- PhotonShuffleMapStage (4)
                                          +- PhotonShuffleExchangeSink (3)
                                             +- PhotonGroupingAgg (2)
                                                +- PhotonScan parquet ocean_watch.raw

#### Ficha 3a · Buques distintos por día

**Resultado.** 31.871 `MMSI` distintos en la semana.

| Día | `count_distinct` | `approx_count_distinct` | Error |
| --- | --- | --- | --- |
| 2023-06-01 | 20.448 | 21.782 | +6,52% |
| 2023-06-02 | 21.153 | 23.182 | +9,59% |
| 2023-06-03 | 20.505 | 20.877 | +1,81% |
| 2023-06-04 | 19.720 | 20.275 | +2,81% |
| 2023-06-05 | 19.615 | 20.358 | +3,79% |
| 2023-06-06 | 19.717 | 20.288 | +2,90% |
| 2023-06-07 | 20.086 | 21.184 | +5,47% |

**Lectura del plan.** `count_distinct` agrupado por `day` se ejecuta en dos agregaciones encadenadas. La primera elimina los pares repetidos (`day`, `MMSI`) y la segunda cuenta por día. Cada agregación tiene su propio intercambio, y el estado de la primera crece con el número de buques distintos. `approx_count_distinct` se resuelve en una sola agregación con un sketch HyperLogLog++ de tamaño fijo por día. El plan muestra un solo intercambio, y lo que viaja por la red son los sketches, no los `MMSI`.

**Lectura del resultado.** El aproximado sobrestima los 7 días, entre +1,81% y +9,59%. La función usa por defecto un error relativo estándar (`rsd`) de 5%, de modo que el día 2 queda cerca de dos desviaciones. El conteo exacto varía entre 19.615 y 21.153 buques en la semana: el error del aproximado es del mismo tamaño que la variación real entre días.

**Decisión técnica. En producción se usa `count_distinct`.** El operador compara el conteo día contra día, y un error de hasta 10% taparía justamente las variaciones que quiere ver. Con 31.871 `MMSI` en la semana, el estado de la deduplicación es pequeño y el segundo intercambio es barato. El aproximado se justifica cuando la cardinalidad llega a millones o cuando la consulta cubre ventanas largas. En ese caso conviene bajar el error, por ejemplo con `approx_count_distinct("MMSI", 0.01)`, a cambio de sketches más grandes.

### 3b. Top 10 tipos de buque por número de posiciones y velocidad media

In [0]:
VESSEL_TYPE_RANGES = [
    (0, 0, "Not available"),
    (1, 19, "Reserved"),
    (20, 29, "Wing in ground"),
    (30, 30, "Fishing"),
    (31, 32, "Towing"),
    (33, 33, "Dredging or underwater ops"),
    (34, 34, "Diving ops"),
    (35, 35, "Military ops"),
    (36, 36, "Sailing"),
    (37, 37, "Pleasure craft"),
    (38, 39, "Reserved"),
    (40, 49, "High speed craft"),
    (50, 50, "Pilot vessel"),
    (51, 51, "Search and rescue"),
    (52, 52, "Tug"),
    (53, 53, "Port tender"),
    (54, 54, "Anti-pollution equipment"),
    (55, 55, "Law enforcement"),
    (56, 57, "Spare, local vessel"),
    (58, 58, "Medical transport"),
    (59, 59, "Noncombatant ship"),
    (60, 69, "Passenger"),
    (70, 79, "Cargo"),
    (80, 89, "Tanker"),
    (90, 99, "Other type"),
]

VESSEL_TYPE_TABLE = f"{CATALOG}.{SCHEMA}.vessel_type_catalog"

vessel_types = spark.createDataFrame(
    [(code, name) for low, high, name in VESSEL_TYPE_RANGES for code in range(low, high + 1)],
    "VesselType INT, vessel_category STRING",
)
(
    vessel_types.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(VESSEL_TYPE_TABLE)
)
spark.sql(f"""
    COMMENT ON TABLE {VESSEL_TYPE_TABLE} IS
    'Catalogo de tipos de buque AIS (ITU-R M.1371, documentado por NOAA Marine Cadastre). Una fila por codigo de VesselType'
""")
spark.sql(f"ALTER TABLE {VESSEL_TYPE_TABLE} ALTER COLUMN VesselType COMMENT 'Codigo AIS de tipo de buque, 0 a 99'")
spark.sql(f"ALTER TABLE {VESSEL_TYPE_TABLE} ALTER COLUMN vessel_category COMMENT 'Categoria del codigo segun el catalogo AIS'")
print(f"{VESSEL_TYPE_TABLE}: {vessel_types.count()} codigos")

ocean_watch.raw.vessel_type_catalog: 100 codigos


In [0]:
from pyspark.sql import Window

ais = spark.table(TABLE)
vessel_catalog = spark.table(VESSEL_TYPE_TABLE)

top_types = (
    ais
    .join(F.broadcast(vessel_catalog), "VesselType", "left")
    .withColumn(
        "vessel_category",
        F.coalesce(
            F.col("vessel_category"),
            F.when(F.col("VesselType").isNull(), F.lit("Sin tipo reportado")).otherwise(F.lit("Fuera del catalogo")),
        ),
    )
    .groupBy("vessel_category")
    .agg(
        F.count("*").alias("posiciones"),
        F.count_distinct("MMSI").alias("buques"),
        F.round(F.avg(F.when(F.col("SOG") != SOG_NO_DISPONIBLE, F.col("SOG"))), 2).alias("sog_medio_nudos"),
    )
    .withColumn("pct_posiciones", F.round(F.col("posiciones") / F.sum("posiciones").over(Window.partitionBy()) * 100, 2))
    .orderBy(F.desc("posiciones"))
    .limit(10)
)

display(top_types)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


vessel_category,posiciones,buques,sog_medio_nudos,pct_posiciones
Towing,16657406,3476,1.68,27.52
Pleasure craft,14476696,12934,1.35,23.92
Passenger,5008900,1774,4.11,8.27
Cargo,4368575,1800,6.46,7.22
Other type,4275801,1678,1.73,7.06
Fishing,4010188,2192,2.24,6.62
Sailing,3876264,4243,1.63,6.4
Tanker,2074793,954,5.94,3.43
Tug,1921565,482,1.72,3.17
"Spare, local vessel",1357965,252,1.9,2.24


In [0]:
top_types.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (25)
+- == Initial Plan ==
   PhotonResultStage (24)
   +- PhotonColumnarToRow (23)
      +- PhotonTopK (22)
         +- PhotonWindow (21)
            +- PhotonShuffleExchangeSource (20)
               +- PhotonShuffleMapStage (19)
                  +- PhotonShuffleExchangeSink (18)
                     +- PhotonGroupingAgg (17)
                        +- PhotonShuffleExchangeSource (16)
                           +- PhotonShuffleMapStage (15)
                              +- PhotonShuffleExchangeSink (14)
                                 +- PhotonGroupingAgg (13)
                                    +- PhotonGroupingAgg (12)
                                       +- PhotonShuffleExchangeSource (11)
                                          +- PhotonShuffleMapStage (10)
                                             +- PhotonShuffleExchangeSink (9)
                                                +- PhotonGroupingAgg (8)
                               

#### Ficha 3b · Tipos de buque con más tráfico

**Resultado.** Top 10 de categorías del catálogo AIS por número de posiciones.

| Categoría | Posiciones | % del total | Buques | `SOG` medio (nudos) |
| --- | --- | --- | --- | --- |
| Towing | 16.657.406 | 27,52% | 3.476 | 1,68 |
| Pleasure craft | 14.476.696 | 23,92% | 12.934 | 1,35 |
| Passenger | 5.008.900 | 8,27% | 1.774 | 4,11 |
| Cargo | 4.368.575 | 7,22% | 1.800 | 6,46 |
| Other type | 4.275.801 | 7,06% | 1.678 | 1,73 |
| Fishing | 4.010.188 | 6,62% | 2.192 | 2,24 |
| Sailing | 3.876.264 | 6,40% | 4.243 | 1,63 |
| Tanker | 2.074.793 | 3,43% | 954 | 5,94 |
| Tug | 1.921.565 | 3,17% | 482 | 1,72 |
| Spare, local vessel | 1.357.965 | 2,24% | 252 | 1,90 |

`Towing` y `Pleasure craft` suman el 51,43% de las posiciones. Un buque puede cambiar de tipo declarado durante la semana, por eso la suma de la columna `Buques` supera los 31.871 `MMSI` distintos.

**Lectura 1. Las categorías de carga se mueven más rápido.** `Cargo` promedia 6,46 nudos y `Tanker` 5,94, frente a 1,68 de `Towing` y 1,35 de `Pleasure craft`. La velocidad media incluye las posiciones con el buque detenido o amarrado, que son la mayoría en las categorías de recreo y remolque.

**Lectura 2. El catálogo cambia el ranking.** Los rangos 60 a 69, 70 a 79 y 80 a 89 agrupan varios códigos bajo una sola categoría. Frente al conteo por código, `Cargo` pasa de 3.837.926 posiciones (solo el código 70) a 4.368.575 y `Tanker` de 1.789.168 (código 80) a 2.074.793. `Pleasure craft` es la categoría con más buques distintos (12.934), aunque `Towing` genera más posiciones: los remolcadores transmiten con más frecuencia.

**Lectura del plan.** El catálogo tiene 100 filas y entra con `BroadcastHashJoin`: se copia a cada ejecutor y las 60,5 millones de filas no se redistribuyen para el join. El único intercambio grande es el de la agregación por categoría. `count_distinct("MMSI")` agrega una segunda etapa de agregación, primero por (`vessel_category`, `MMSI`) y luego por categoría, con su propio intercambio: es el costo de reportar buques además de posiciones.

**Decisión técnica 1. Catálogo como tabla gobernada.** `ocean_watch.raw.vessel_type_catalog` se escribe con comentario de tabla y de columnas, para que el resto del notebook y la Entrega 2 usen la misma clasificación. Los códigos fuera de 0 a 99 quedan como `Fuera del catalogo` en vez de perderse en el join.

**Decisión técnica 2. `SOG = 102,3` queda fuera del promedio.** Es el valor de no disponible del estándar (R3), no una velocidad. Incluirlo subiría el promedio de las categorías con más equipos sin sensor.

### 3c. Top 10 buques por distancia recorrida en la semana (haversine)

In [0]:
KNOT_KMH = 1.852
SEGMENTO_KMH_MAX = SOG_MAX * KNOT_KMH
R_TIERRA_KM = 6371.0

w = Window.partitionBy("MMSI").orderBy("BaseDateTime")

posiciones = (
    spark.table(TABLE)
    .where(MMSI_VALIDO & F.col("LAT").between(-90, 90) & F.col("LON").between(-180, 180))
    .select("MMSI", "BaseDateTime", "LAT", "LON", "VesselName")
    .withColumn("ts_anterior", F.lag("BaseDateTime").over(w))
    .where(F.col("ts_anterior").isNull() | (F.col("BaseDateTime") > F.col("ts_anterior")))
)

segmentos = (
    posiciones
    .withColumn("lat_anterior", F.lag("LAT").over(w))
    .withColumn("lon_anterior", F.lag("LON").over(w))
    .withColumn("ts_anterior", F.lag("BaseDateTime").over(w))
    .where(F.col("ts_anterior").isNotNull())
    .withColumn("horas", (F.unix_timestamp("BaseDateTime") - F.unix_timestamp("ts_anterior")) / 3600)
    .withColumn(
        "distancia_km",
        2 * R_TIERRA_KM * F.asin(F.sqrt(
            F.pow(F.sin(F.radians(F.col("LAT") - F.col("lat_anterior")) / 2), 2)
            + F.cos(F.radians("lat_anterior")) * F.cos(F.radians("LAT"))
            * F.pow(F.sin(F.radians(F.col("LON") - F.col("lon_anterior")) / 2), 2)
        )),
    )
    .withColumn("segmento_valido", F.col("distancia_km") / F.col("horas") <= SEGMENTO_KMH_MAX)
)

In [0]:
top_10_distancia = (
    segmentos
    .groupBy("MMSI")
    .agg(
        F.round(F.sum(F.when(F.col("segmento_valido"), F.col("distancia_km")).otherwise(0.0)), 1).alias("distancia_km"),
        F.round(F.sum("distancia_km"), 1).alias("distancia_sin_filtro_km"),
        F.sum(F.col("segmento_valido").cast("int")).alias("segmentos_validos"),
        F.sum((~F.col("segmento_valido")).cast("int")).alias("segmentos_descartados"),
        F.first("VesselName", ignorenulls=True).alias("vessel_name"),
    )
    .orderBy(F.desc("distancia_km"))
    .limit(10)
)

display(top_10_distancia)

MMSI,distancia_km,distancia_sin_filtro_km,segmentos_validos,segmentos_descartados,vessel_name
367638030,5770.9,12670.2,2865,8,JUSTIN PAUL ECKSTEIN
367003380,4492.9,5338.0,2944,5,JEAN ANNE
310531000,4424.4,5405.5,2619,9,EMERALD PRINCESS
369355000,4396.9,4406.3,2308,8,GEORGE III
366629000,4392.0,6177.3,3072,16,HORIZON SPIRIT
311000396,4145.0,4145.0,2803,0,HARMONY OF THE SEAS
311001223,4073.2,4073.2,2258,0,CARNIVAL CELEBRATION
338796000,3986.9,3986.9,2628,0,MATSONIA
367799380,3973.3,3973.3,2915,0,TAINO
311000341,3942.5,3942.5,2634,0,NORWEGIAN ESCAPE


In [0]:
resumen_3c = segmentos.agg(
    F.count("*").alias("segmentos"),
    F.sum((~F.col("segmento_valido")).cast("long")).alias("segmentos_descartados"),
    F.round(F.sum(F.when(F.col("segmento_valido"), F.col("distancia_km"))), 0).alias("km_validos"),
    F.round(F.sum(F.when(~F.col("segmento_valido"), F.col("distancia_km"))), 0).alias("km_descartados"),
)
display(resumen_3c)
print(f"Cota fisica de un buque en 7 dias a {SOG_MAX:.0f} nudos: {7 * 24 * SEGMENTO_KMH_MAX:,.0f} km")

segmentos,segmentos_descartados,km_validos,km_descartados
60450212,52202,7340648.0,3.5587703E7


Cota fisica de un buque en 7 dias a 35 nudos: 10,890 km


In [0]:
top_10_distancia.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (21)
+- == Initial Plan ==
   PhotonResultStage (20)
   +- PhotonColumnarToRow (19)
      +- PhotonTopK (18)
         +- PhotonShuffleExchangeSource (17)
            +- PhotonShuffleMapStage (16)
               +- PhotonShuffleExchangeSink (15)
                  +- PhotonTopK (14)
                     +- PhotonGroupingAgg (13)
                        +- PhotonProject (12)
                           +- PhotonProject (11)
                              +- PhotonFilter (10)
                                 +- PhotonWindow (9)
                                    +- PhotonProject (8)
                                       +- PhotonFilter (7)
                                          +- PhotonWindow (6)
                                             +- PhotonSort (5)
                                                +- PhotonShuffleExchangeSource (4)
                                                   +- PhotonShuffleMapStage (3)
                              

#### Ficha 3c · Buques con más distancia recorrida

**Resultado.** Los 10 buques con más distancia en la semana.

| `MMSI` | Buque | Distancia (km) | Sin filtro (km) | Segmentos descartados |
| --- | --- | --- | --- | --- |
| 367638030 | JUSTIN PAUL ECKSTEIN | 5.770,9 | 12.670,2 | 8 |
| 367003380 | JEAN ANNE | 4.492,9 | 5.338,0 | 5 |
| 310531000 | EMERALD PRINCESS | 4.424,4 | 5.405,5 | 9 |
| 369355000 | GEORGE III | 4.396,9 | 4.406,3 | 8 |
| 366629000 | HORIZON SPIRIT | 4.392,0 | 6.177,3 | 16 |
| 311000396 | HARMONY OF THE SEAS | 4.145,0 | 4.145,0 | 0 |
| 311001223 | CARNIVAL CELEBRATION | 4.073,2 | 4.073,2 | 0 |
| 338796000 | MATSONIA | 3.986,9 | 3.986,9 | 0 |
| 367799380 | TAINO | 3.973,3 | 3.973,3 | 0 |
| 311000341 | NORWEGIAN ESCAPE | 3.942,5 | 3.942,5 | 0 |

Todos quedan por debajo de la cota física de 10.890 km. En toda la tabla se descartaron 52.202 de 60.450.212 segmentos (0,09%), pero esos segmentos sumaban 35.587.703 km, el 82,9% de la distancia sin filtro.

**Lectura 1. El ranking es coherente con el tipo de buque.** Del segundo al décimo lugar hay cruceros (EMERALD PRINCESS, HARMONY OF THE SEAS, CARNIVAL CELEBRATION, NORWEGIAN ESCAPE) y buques de carga de línea regular (MATSONIA, TAINO, HORIZON SPIRIT, JEAN ANNE), con entre 3.940 y 4.490 km: unos 13 nudos sostenidos toda la semana.

**Lectura 2. El primer lugar merece revisión.** JUSTIN PAUL ECKSTEIN conserva 5.771 km, unos 18,5 nudos sostenidos durante 7 días. Los 8 segmentos descartados sumaban 6.899 km, así que este `MMSI` tiene saltos grandes y probablemente también ruido de posición por debajo del umbral. Un umbral por tipo de buque, registrado como pendiente en 2.6, lo resolvería.

**Hallazgo. La primera versión daba distancias imposibles.** Sin filtros, el primer buque sumaba 7.931.112 km en 7 días, unas 198 vueltas a la Tierra. A 35 nudos sostenidos, un buque recorre como máximo unos 10.890 km en una semana. La causa son saltos entre posiciones consecutivas: un reporte con coordenadas erróneas, o dos transmisores que comparten el mismo `MMSI`, generan segmentos de miles de kilómetros en segundos.

**Decisión técnica 1. Filtros previos.** Se excluyen los `MMSI` con formato inválido (R1), las coordenadas fuera de rango (R2) y los reportes repetidos para el mismo `MMSI` e instante (R9), que dan segmentos de cero horas.

**Decisión técnica 2. Velocidad implícita por segmento.** Cada segmento entre dos posiciones consecutivas del mismo buque se divide entre el tiempo transcurrido. Si la velocidad implícita supera `SOG_MAX` (35 nudos, 64,8 km/h), el segmento se descarta. Es el mismo umbral de la regla R3, aplicado a la trayectoria en vez de al campo `SOG` declarado.

**Decisión técnica 3. Un solo intercambio.** Los duplicados por llave se eliminan con `lag` sobre la misma ventana (`MMSI` ordenado por `BaseDateTime`) en vez de `dropDuplicates`. Las dos ventanas comparten partición y orden, así que el plan muestra un único intercambio por `MMSI` y un único `Sort`, seguido de los operadores `Window`. La agregación por `MMSI` reutiliza esa misma partición y tampoco agrega intercambio. Con `dropDuplicates` habría un segundo intercambio sobre las 60,5 millones de filas.

**Limitaciones.** Al descartar un salto se pierden los dos segmentos que lo rodean, así que la distancia de un buque con saltos queda levemente subestimada. La distancia entre reportes es la de gran círculo, una cota inferior de la trayectoria real cuando hay huecos de transmisión.

### 3d. Top 10 celdas de tráfico (H3 res. 8 o rejilla lat/lon) y cruce con el World Port Index

In [0]:
%pip install h3
import h3
import pandas as pd
from pyspark.sql.functions import pandas_udf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
@pandas_udf("string")
def to_h3(lat: pd.Series, lon: pd.Series) -> pd.Series:
    return pd.Series([h3.latlng_to_cell(la, lo, 8) for la, lo in zip(lat, lon)])

@pandas_udf("string")
def to_h3_parent(cell: pd.Series) -> pd.Series:
    return pd.Series([h3.cell_to_parent(c, 6) if c is not None else None for c in cell])

In [0]:
ais = spark.table("ocean_watch.raw.ais")
top_cells = (
    ais
    .where("LAT IS NOT NULL AND LON IS NOT NULL")
    .withColumn("h3_cell", to_h3("LAT", "LON"))
    .groupBy("h3_cell")
    .agg(F.count("*").alias("positions"))
    .orderBy(F.desc("positions"))
    .limit(10)
    .withColumn("h3_parent", to_h3_parent("h3_cell"))
)

In [0]:
wpi = spark.table("ocean_watch.raw.world_port_index")

wpi_grid = (
    wpi
    .withColumn("h3_cell", to_h3("LAT", "LON"))
    .withColumn("h3_parent", to_h3_parent("h3_cell"))
    .groupBy("h3_parent")
    .agg(F.collect_set("port_name").alias("ports"))
)

In [0]:
top_10 = (
    top_cells
    .join(wpi_grid, "h3_parent", "left")
    .orderBy(F.desc("positions"))
    .select("h3_cell", "positions", "ports")
)

display(top_10)

h3_cell,positions,ports
8828d555a1fffff,235268,null
8828d54715fffff,184414,null
8829a411d7fffff,180100,List(SAN DIEGO)
8828d17b59fffff,123785,null
8829a19a35fffff,115457,null
88446e0359fffff,114645,null
8829127827fffff,109522,null
8844a13b51fffff,103934,List(PORT EVERGLADES)
8828d5476bfffff,101039,null
8829a411a9fffff,100848,List(SAN DIEGO)


In [0]:
top_10.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (57)
+- == Initial Plan ==
   PhotonResultStage (56)
   +- PhotonColumnarToRow (55)
      +- PhotonSort (54)
         +- PhotonProject (53)
            +- PhotonBroadcastHashJoin LeftOuter (52)
               :- PhotonGlobalLimit (27)
               :  +- PhotonSort (26)
               :     +- PhotonShuffleExchangeSource (25)
               :        +- PhotonShuffleMapStage (24)
               :           +- PhotonShuffleExchangeSink (23)
               :              +- PhotonLocalLimit (22)
               :                 +- PhotonProject (21)
               :                    +- PhotonArrowBatchSource (20)
               :                       +- ArrowEvalPython (19)
               :                          +- PhotonArrowResultStage (18)
               :                             +- PhotonArrowBatchSink (17)
               :                                +- PhotonLocalLimit (16)
               :                                   +- Phot

#### Ficha 3d · Celdas con más tráfico y cruce con el World Port Index

**Resultado.** Las 10 celdas H3 de resolución 8 con más posiciones.

| # | Celda | Posiciones | Centro | Zona | Puerto WPI |
| --- | --- | --- | --- | --- | --- |
| 1 | `8828d555a1fffff` | 235.268 | 47,63 N, 122,39 O | Seattle, Elliott Bay | |
| 2 | `8828d54715fffff` | 184.414 | 47,68 N, 122,41 O | Seattle, Shilshole Bay | |
| 3 | `8829a411d7fffff` | 180.100 | 32,72 N, 117,23 O | San Diego, Shelter Island | SAN DIEGO |
| 4 | `8828d17b59fffff` | 123.785 | 48,76 N, 122,50 O | Bellingham | |
| 5 | `8829a19a35fffff` | 115.457 | 33,98 N, 118,45 O | Marina del Rey, Los Ángeles | |
| 6 | `88446e0359fffff` | 114.645 | 29,97 N, 93,86 O | Port Arthur, Texas | |
| 7 | `8829127827fffff` | 109.522 | 34,25 N, 119,26 O | Ventura | |
| 8 | `8844a13b51fffff` | 103.934 | 26,10 N, 80,16 O | Fort Lauderdale | PORT EVERGLADES |
| 9 | `8828d5476bfffff` | 101.039 | 47,66 N, 122,37 O | Seattle, Salmon Bay | |
| 10 | `8829a411a9fffff` | 100.848 | 32,73 N, 117,19 O | San Diego, bahía | SAN DIEGO |

El centro y la zona se obtienen con `h3.cell_to_latlng` sobre cada celda.

Solo 3 de las 10 celdas caen en un puerto del World Port Index: dos en SAN DIEGO y una en PORT EVERGLADES. San Diego es la zona usada como propósito de consulta en la sección 4.

**Lectura 1. El tráfico se concentra en marinas y aguas interiores.** Seattle aporta 3 de las 10 celdas y San Diego 2. Varias celdas son marinas de recreo (Shilshole Bay, Marina del Rey, Ventura), coherente con 3b: `Pleasure craft` y `Towing` generan la mitad de las posiciones.

**Lectura 2. Hay puertos que el cruce no encuentra.** Seattle y Port Arthur son puertos comerciales y es probable que figuren en el World Port Index, pero sus celdas no se cruzan. El punto de referencia del puerto cae a varios kilómetros de donde se concentran los buques y, en esos casos, en otra celda de resolución 6. Es la limitación descrita al final de esta ficha.

**Lectura del plan.** Las funciones `to_h3` y `to_h3_parent` son UDF de pandas y aparecen como `ArrowEvalPython`: cada fila se serializa en lotes Arrow hacia un proceso Python. Es el paso más costoso de la consulta. `limit(10)` se aplica antes de calcular la celda madre, así que `to_h3_parent` corre solo sobre 10 filas. El cruce con el WPI une dos tablas pequeñas y se resuelve con `BroadcastHashJoin`.

**Decisión técnica 1. Resolución 8 para el tráfico y 6 para el cruce.** La resolución 8 (celdas de unos 0,7 km²) es la que pide el enunciado y separa canales y dársenas. El punto del WPI es la referencia del puerto y cae a kilómetros de donde se mueven los buques, así que casi nunca comparte celda de resolución 8. Ambas celdas se llevan a su madre de resolución 6 (unos 36 km²), que cubre el área portuaria.

**Decisión técnica 2. `left join`.** Conserva las 10 celdas aunque no tengan puerto, para mostrar qué parte del tráfico no está asociada a un puerto del índice.

**Limitaciones y mejora.** Un puerto justo al otro lado del borde de una celda de resolución 6 no se cruza. Se resolvería con los vecinos de la celda (`h3.grid_disk`). Las funciones H3 nativas de Databricks (`h3_longlatash3string`, `h3_toparent`) evitarían la ejecución en Python.

### 3e. Proporción de buques que transmitieron los 7 días y ubicación de los "visitantes de un solo día"

In [0]:
ais = spark.table("ocean_watch.raw.ais")

vessel_days = (
    ais
    .groupBy("MMSI")
    .agg(F.count_distinct("day").alias("days_active"))
)

days_active = (
    vessel_days
    .groupBy("days_active")
    .agg(F.count("*").alias("vessels"))
    .orderBy("days_active")
)

display(days_active)

days_active,vessels
1,5996
2,4391
3,2896
4,2205
5,2007
6,1709
7,12667


In [0]:
single_day_mmsis = vessel_days.where(F.col("days_active") == 1).select("MMSI")

single_day_locations = (
    ais
    .join(F.broadcast(single_day_mmsis), "MMSI")
    .where("LAT IS NOT NULL AND LON IS NOT NULL")
    .withColumn("h3_cell", to_h3("LAT", "LON"))
    .groupBy("h3_cell")
    .agg(
        F.count("*").alias("positions"),
        F.count_distinct("MMSI").alias("vessels"),
        F.round(F.avg("LAT"), 2).alias("lat"),
        F.round(F.avg("LON"), 2).alias("lon"),
    )
    .orderBy(F.desc("vessels"))
    .limit(10)
)

display(single_day_locations)

h3_cell,positions,vessels,lat,lon
882aa802cdfffff,266,49,38.97,-76.46
8828d54715fffff,1319,47,47.68,-122.41
882aa80213fffff,239,43,38.97,-76.48
882aa80217fffff,811,42,38.97,-76.48
8844a13b2dfffff,325,41,26.1,-80.12
882aa80211fffff,273,41,38.98,-76.48
882aa8021bfffff,245,39,38.98,-76.47
8844a13b67fffff,179,38,26.1,-80.12
882aa802e9fffff,510,35,38.97,-76.48
8828d54711fffff,132,34,47.68,-122.42


In [0]:
single_day_locations.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (42)
+- == Initial Plan ==
   PhotonResultStage (41)
   +- PhotonColumnarToRow (40)
      +- PhotonTopK (39)
         +- PhotonShuffleExchangeSource (38)
            +- PhotonShuffleMapStage (37)
               +- PhotonShuffleExchangeSink (36)
                  +- PhotonTopK (35)
                     +- PhotonGroupingAgg (34)
                        +- PhotonShuffleExchangeSource (33)
                           +- PhotonShuffleMapStage (32)
                              +- PhotonShuffleExchangeSink (31)
                                 +- PhotonGroupingAgg (30)
                                    +- PhotonGroupingAgg (29)
                                       +- PhotonShuffleExchangeSource (28)
                                          +- PhotonShuffleMapStage (27)
                                             +- PhotonShuffleExchangeSink (26)
                                                +- PhotonGroupingAgg (25)
                               

#### Ficha 3e · Buques de toda la semana y visitantes de un solo día

**Resultado.**

| Días con transmisión | Buques | % de los 31.871 |
| --- | --- | --- |
| 1 | 5.996 | 18,81% |
| 2 | 4.391 | 13,78% |
| 3 | 2.896 | 9,09% |
| 4 | 2.205 | 6,92% |
| 5 | 2.007 | 6,30% |
| 6 | 1.709 | 5,36% |
| 7 | 12.667 | 39,74% |

El 39,74% de los buques transmitió los 7 días. El 18,81% aparece un solo día.

**Lectura 1. Dónde están los visitantes de un solo día.** Las 10 celdas con más visitantes de un solo día están en tres zonas: 6 celdas en la bahía de Chesapeake frente a Annapolis (38,97 N, 76,48 O), 2 en Seattle (47,68 N, 122,41 O) y 2 en Fort Lauderdale (26,10 N, 80,12 O). Las tres son polos de navegación de recreo, lo que apunta a embarcaciones que salen un día y no vuelven a transmitir en la semana.

**Lectura 2. La celda de Seattle es a la vez de alto tráfico.** La celda `8828d54715fffff` es la segunda con más posiciones de toda la semana en 3d (184.414) y la segunda con más visitantes de un solo día (47). Combina una flota residente que transmite todos los días con muchas visitas puntuales.

**Lectura del plan.** `vessel_days` se calcula con una agregación por `MMSI` y se reutiliza en las dos celdas. En serverless no hay `cache()`, así que el plan de la segunda celda vuelve a calcular esa agregación. Los 5.996 `MMSI` de un solo día entran al join con `F.broadcast`, y el plan muestra `BroadcastHashJoin` en lugar de redistribuir las 60,5 millones de filas por `MMSI`.

**Decisión técnica. Broadcast explícito.** La lista de visitantes es pequeña pero sale de una agregación, así que Spark no conoce su tamaño al planificar. El `F.broadcast` fija el tipo de join desde el plan inicial en vez de depender de que la ejecución adaptativa lo corrija en tiempo de ejecución.

# 4. Almacenamiento óptimo para un propósito

**El tablero diario del operador de tráfico: todas las posiciones de un día dentro de una zona marítima.**

El operador selecciona un cuadrante que corresponde a un puerto en particular y revisa el tráfico de una fecha en especifico. La consulta que representa ese trabajo es:

```sql
SELECT * FROM ais
WHERE day = '2023-06-04' -- por ejemplo
  AND LAT BETWEEN 32.60 AND 32.80 -- 
  AND LON BETWEEN -117.30 AND -117.10
```

La zona corresponde a la bahía de San Diego. La pregunta 3d identifica ese punto como uno de los de mayor tráfico del dataset, con las celdas H3 `8829a411d7fffff` (180.100 posiciones) y `8829a411a9fffff` (100.848 posiciones), las dos cruzadas con el puerto SAN DIEGO del World Port Index.

**Por qué este propósito.** Esta consulta representa el uso real del caso OceanWatch. Ademas que permite medir la mejora con evidencia clara, porque el filtro espacial deja fuera casi toda la tabla.

**Qué queda fuera del propósito.** La reconstrucción de la trayectoria de un buque filtra por `MMSI` y por rango de tiempo. Ese patrón pide un layout distinto. La sección 4.6 declara el costo que asume esa consulta con el layout elegido.

### Diseño

| Decisión | Elección | Razón |
| --- | --- | --- |
| Formato de archivo | Parquet | Columnar, comprimido, con estadísticas por grupo de filas. El CSV obliga a leer y convertir todas las columnas. |
| Formato de tabla | Delta | El enunciado pide el efecto de `OPTIMIZE`, que es una operación de Delta. Delta aporta además el registro de transacciones y el salto de archivos por estadísticas. |
| Partición | `day` | El filtro por fecha es fijo en el propósito. R7 demuestra que `day` reproduce el corte del origen. 2.1 demuestra que las 7 particiones tienen tamaño parejo, entre 8,04 y 9,05 millones de filas. |
| Agrupación dentro de la partición | `OPTIMIZE ... ZORDER BY (LAT, LON)` | El filtro espacial necesita que las posiciones cercanas queden en el mismo archivo. Z-order ordena por las dos columnas a la vez y mejora el salto de archivos. |

**Por qué no `CLUSTER BY` líquido.** El agrupamiento líquido sustituye a la partición y no admite las dos técnicas juntas. El propósito declarado filtra siempre por una fecha exacta, y la partición por `day` descarta 6 de las 7 particiones sin abrir un solo archivo. La celda 4.7 deja la variante líquida disponible para comparar, con la bandera `EJECUTAR_LIQUID`.

In [0]:
# 4.3 Parámetros de la sección.
# Las variantes se escriben en el Volume, de modo que las cuatro se miden
# con el mismo procedimiento y con la misma unidad.
RUTA_OPT = f"{VOLUME_ROOT}/opt"
RUTA_PARQUET = f"{RUTA_OPT}/parquet_dia"
RUTA_DELTA = f"{RUTA_OPT}/delta_dia"
RUTA_DELTA_Z = f"{RUTA_OPT}/delta_dia_zorder"
RUTA_LIQUID = f"{RUTA_OPT}/delta_liquid"

DIA_CONSULTA = "2023-06-04"
LAT_MIN, LAT_MAX = 32.60, 32.80
LON_MIN, LON_MAX = -117.30, -117.10


def consulta_operador(df):
    # La consulta declarada en 4.1: un día y una zona marítima.
    return (
        df.where(F.col("day") == F.lit(DIA_CONSULTA))
          .where(F.col("LAT").between(LAT_MIN, LAT_MAX))
          .where(F.col("LON").between(LON_MIN, LON_MAX))
    )


def medir_ruta(ruta, extension=".parquet"):
    # Recorre el árbol del Volume y suma los bytes de los archivos de datos.
    # Los archivos del registro de transacciones de Delta quedan fuera de la suma.
    total_bytes, archivos = 0, 0
    pendientes = [ruta]
    while pendientes:
        actual = pendientes.pop()
        for f in dbutils.fs.ls(actual):
            if f.name.endswith("/"):
                pendientes.append(f.path)
            elif f.name.endswith(extension):
                total_bytes += f.size
                archivos += 1
    return total_bytes, archivos


def medir_delta(ruta):
    # Tamaño y archivos de la versión vigente de una tabla Delta.
    # Recorrer el directorio contaría también los archivos que OPTIMIZE reemplazó y que siguen en disco hasta un VACUUM.
    detalle = spark.sql(f"DESCRIBE DETAIL delta.`{ruta}`").collect()[0]
    return detalle["sizeInBytes"], detalle["numFiles"]


def archivos_leidos(df):
    # Archivos que aportan filas a la consulta declarada.
    # El enunciado acepta esta medida o el plan de ejecución.
    # _metadata.file_path reemplaza a input_file_name(), que Unity Catalog no admite.
    return (
        consulta_operador(df)
        .select(F.col("_metadata.file_path").alias("archivo"))
        .distinct()
        .count()
    )


print("Zona:", LAT_MIN, LAT_MAX, LON_MIN, LON_MAX, "| Día:", DIA_CONSULTA)

Zona: 32.6 32.8 -117.3 -117.1 | Día: 2023-06-04


### Experimento

Para verificar la optimizacion se escriben 3 variantes. Cada variante parte de la misma tabla `ocean_watch.raw.ais`, con las mismas 60M de filas y la misma cantidad de columnas.

In [0]:
# Variante 1: Parquet particionado por day.
(
    ais.write.mode("overwrite").format("parquet")
    .partitionBy("day")
    .save(RUTA_PARQUET)
)

# Variante 2: Delta particionado por day, sin OPTIMIZE.
(
    ais.write.mode("overwrite").format("delta")
    .partitionBy("day")
    .save(RUTA_DELTA)
)

# Variante 3: copia de la variante 2, para medir el efecto de OPTIMIZE sobre la misma base.
(
    spark.read.format("delta").load(RUTA_DELTA)
    .write.mode("overwrite").format("delta")
    .partitionBy("day")
    .save(RUTA_DELTA_Z)
)

print("Variantes escritas en", RUTA_OPT)

Variantes escritas en /Volumes/ocean_watch/raw/ais_raw/opt


In [0]:
# Medición antes de OPTIMIZE.
bytes_csv, archivos_csv = medir_ruta(CSV_DIR, ".csv")
bytes_parquet, archivos_parquet = medir_ruta(RUTA_PARQUET)
bytes_delta, archivos_delta = medir_delta(RUTA_DELTA)
bytes_z_antes, archivos_z_antes = medir_delta(RUTA_DELTA_Z)

df_parquet = spark.read.format("parquet").load(RUTA_PARQUET)
df_delta = spark.read.format("delta").load(RUTA_DELTA)
df_z = spark.read.format("delta").load(RUTA_DELTA_Z)

leidos_parquet = archivos_leidos(df_parquet)
leidos_delta = archivos_leidos(df_delta)
leidos_z_antes = archivos_leidos(df_z)

filas_consulta = consulta_operador(df_delta).count()

print(f"CSV            : {humano(bytes_csv)} en {archivos_csv} archivos")
print(f"Parquet        : {humano(bytes_parquet)} en {archivos_parquet} archivos")
print(f"Delta          : {humano(bytes_delta)} en {archivos_delta} archivos")
print(f"Delta (copia)  : {humano(bytes_z_antes)} en {archivos_z_antes} archivos")
print()
print(f"Filas que devuelve la consulta declarada: {filas_consulta:,}")
print(f"Archivos leidos Parquet      : {leidos_parquet}")
print(f"Archivos leidos Delta        : {leidos_delta}")
print(f"Archivos leidos Delta (copia): {leidos_z_antes}")

CSV            : 6.0 GB en 7 archivos
Parquet        : 1.8 GB en 42 archivos
Delta          : 1.8 GB en 7 archivos
Delta (copia)  : 1.8 GB en 7 archivos

Filas que devuelve la consulta declarada: 139,844
Archivos leidos Parquet      : 6
Archivos leidos Delta        : 1
Archivos leidos Delta (copia): 1


In [0]:
# Ejecución de OPTIMIZE con Z-order sobre LAT y LON.
# Un día pesa unos 270 MB en Parquet. Con el tamaño objetivo por defecto, OPTIMIZE deja un solo archivo por día
# y el Z-order no tendría archivos que descartar dentro de la partición.
TAMANO_OBJETIVO = "32mb"
spark.sql(f"ALTER TABLE delta.`{RUTA_DELTA_Z}` SET TBLPROPERTIES ('delta.targetFileSize' = '{TAMANO_OBJETIVO}')")

resultado_optimize = spark.sql(
    f"OPTIMIZE delta.`{RUTA_DELTA_Z}` ZORDER BY (LAT, LON)"
)
display(resultado_optimize)

path,metrics
dbfs:/Volumes/ocean_watch/raw/ais_raw/opt/delta_dia_zorder,"List(60, 7, List(19199184, 42623812, 2.7181015216666665E7, 60, 1630860913), List(253266579, 288340130, 2.728078055714286E8, 7, 1909654639), 7, List(minCubeSize(107374182400), List(0, 0), List(7, 1909654639), 0, List(7, 1909654639), 7, null), null, 0, 1, 7, 0, false, 0, 0, 1790342325242, 1790342382161, 8, 7, null, List(0, 0), null, 21, 21, 132566, 0, null, null, 0)"


In [0]:
# Medición después de OPTIMIZE.
bytes_z_despues, archivos_z_despues = medir_delta(RUTA_DELTA_Z)

df_z = spark.read.format("delta").load(RUTA_DELTA_Z)
leidos_z_despues = archivos_leidos(df_z)
filas_z = consulta_operador(df_z).count()

print(f"Delta + ZORDER : {humano(bytes_z_despues)} en {archivos_z_despues} archivos")
print(f"Archivos leidos Delta + ZORDER: {leidos_z_despues}")
print(f"Filas que devuelve la consulta: {filas_z:,} (debe coincidir con {filas_consulta:,})")

Delta + ZORDER : 1.5 GB en 60 archivos
Archivos leidos Delta + ZORDER: 1
Filas que devuelve la consulta: 139,844 (debe coincidir con 139,844)


In [0]:
# Tabla de evidencia. Una fila por variante.
evidencia = spark.createDataFrame(
    [
        ("1. CSV original",      bytes_csv,       archivos_csv,       None,              None),
        ("2. Parquet por day",   bytes_parquet,   archivos_parquet,   leidos_parquet,    None),
        ("3. Delta por day",     bytes_delta,     archivos_delta,     leidos_delta,      None),
        ("4. Delta + ZORDER",    bytes_z_despues, archivos_z_despues, leidos_z_despues,  archivos_z_antes),
    ],
    "variante string, bytes long, archivos_totales long, archivos_leidos long, archivos_antes_de_optimize long",
).withColumn(
    "pct_bytes_vs_csv", F.round(F.col("bytes") * 100.0 / F.lit(bytes_csv), 2)
).withColumn(
    "pct_archivos_leidos", F.round(F.col("archivos_leidos") * 100.0 / F.col("archivos_totales"), 2)
)

display(evidencia)

variante,bytes,archivos_totales,archivos_leidos,archivos_antes_de_optimize,pct_bytes_vs_csv,pct_archivos_leidos
1. CSV original,6481921444,7,null,null,100.0,null
2. Parquet por day,1882649536,42,6,null,29.04,14.29
3. Delta por day,1910261086,7,1,null,29.47,14.29
4. Delta + ZORDER,1630860913,60,1,7,25.16,1.67


In [0]:
# Planes de ejecución de la consulta declarada, antes y después del Z-order.
print("=== Delta particionado, sin ZORDER ===")
consulta_operador(spark.read.format("delta").load(RUTA_DELTA)).explain("formatted")

print("\n=== Delta particionado + ZORDER ===")
consulta_operador(spark.read.format("delta").load(RUTA_DELTA_Z)).explain("formatted")

=== Delta particionado, sin ZORDER ===
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [21]: [MMSI#30149, BaseDateTime#30150, LAT#30151, LON#30152, SOG#30153, COG#30154, Heading#30155, VesselName#30156, IMO#30157, CallSign#30158, VesselType#30159, Status#30160, Length#30161, Width#30162, Draft#30163, Cargo#30164, TransceiverClass#30165, _corrupt_record#30166, source_file#30167, ingested_at#30169, day#30168]
DictionaryFilters: [((LON#30152 >= -117.3) AND (LON#30152 <= -117.1)), ((LAT#30151 >= 32.6) AND (LAT#30151 <= 32.8))]
Location: PreparedDeltaFileIndex [dbfs:/Volumes/ocean_watch/raw/ais_raw/opt/delta_dia]
PartitionFilters: [isnotnull(day#30168), (day#30168 = 2023-06-04)]
ReadSchema: struct<MMSI:string,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double,COG:double,Heading:double,VesselName:string,IMO:string,CallSign:string,VesselType:int,Status:int,Length:double,Width:do

### 4.5 Evidencia

La tabla de la celda anterior reúne las tres medidas que pide el enunciado: bytes en disco, archivos totales y archivos leídos por la consulta declarada.

Consulta declarada: `day = '2023-06-04'` y la zona de la bahía de San Diego. Devuelve 139.844 filas en todas las variantes.

| Variante | Bytes | % frente al CSV | Archivos totales | Archivos leídos | % de archivos leídos |
| --- | --- | --- | --- | --- | --- |
| 1. CSV original | 6.481.921.444 (6,04 GB) | 100% | 7 | | |
| 2. Parquet por `day` | 1.882.649.536 (1,75 GB) | 29,04% | 42 | 6 | 14,29% |
| 3. Delta por `day` | 1.910.261.086 (1,78 GB) | 29,47% | 7 | 1 | 14,29% |
| 4. Delta + `ZORDER` | 1.630.860.913 (1,52 GB) | 25,16% | 60 | 1 | 1,67% |

**Lectura 1. El formato columnar reduce el tamaño a menos de un tercio.** Parquet y Delta ocupan el 29% del CSV. El Z-order baja el tamaño otro 14,6% frente a Delta sin ordenar, porque las posiciones cercanas quedan juntas y comprimen mejor.

**Lectura 2. La partición por `day` descarta 6 de 7 días.** Parquet lee los 6 archivos del día consultado y Delta su único archivo. En los dos casos es 1/7 de la tabla (14,29%). Parquet deja 6 archivos por día y Delta 1, porque el escritor Delta de Databricks consolida los archivos de cada partición.

**Lectura 3. El Z-order reduce los datos leídos entre 6 y 15 veces.** Sin Z-order, la consulta abre el archivo completo del día, de entre 253 y 288 MB según las métricas de `OPTIMIZE`. Con Z-order, abre 1 de los 60 archivos, de entre 19 y 43 MB. El número de archivos leídos es 1 en los dos casos, por eso la comparación relevante es el porcentaje de archivos: de 14,29% a 1,67%.

**Tamaño objetivo de archivo.** Antes de `OPTIMIZE` la variante 4 fija `delta.targetFileSize = 32mb`. Un día ocupa unos 270 MB en Parquet; con el tamaño objetivo por defecto, `OPTIMIZE` compacta cada día en un solo archivo y la reducción de archivos leídos vendría de la compactación, no del Z-order. Con 32 MB, `OPTIMIZE` reemplazó los 7 archivos por 60 (unos 27 MB en promedio), ordenados por `LAT` y `LON`, y la consulta del tablero lee solo el que cubre la zona.

**Qué mide cada columna.** `bytes` y `archivos_totales` de las variantes Delta salen de `DESCRIBE DETAIL`, que cuenta solo la versión vigente de la tabla. `archivos_leidos` cuenta los archivos que aportan al menos una fila a la consulta (`_metadata.file_path`, equivalente a `INPUT_FILE_NAME`). Los archivos que Spark abre y descarta por estadísticas no entran en esa medida; aparecen en las métricas del escaneo del perfil de la consulta.

**Qué buscar en el plan.** El plan de la consulta muestra dos filtros. `PartitionFilters` contiene la condición sobre `day` y descarta 6 de las 7 particiones antes de abrir un archivo. `PushedFilters` contiene las condiciones sobre `LAT` y `LON`, que Parquet aplica con las estadísticas de mínimo y máximo de cada grupo de filas. `Z-order` mejora el segundo filtro, porque agrupa las posiciones cercanas en los mismos archivos y estrecha el rango de esas estadísticas.

In [0]:
# 4.6 Contraste con el propósito no declarado.
# Trayectoria de un buque: filtra por MMSI y por rango de tiempo.
# El buque de ejemplo pasa por la zona del tablero y es el que más se desplaza ese día,
# para que su trayectoria cruce varias zonas del Z-order.
df_base = spark.read.format("delta").load(RUTA_DELTA)
buques_zona = consulta_operador(df_base).select("MMSI").distinct()
MMSI_EJEMPLO = (
    df_base.where(F.col("day") == F.lit(DIA_CONSULTA))
    .join(F.broadcast(buques_zona), "MMSI")
    .groupBy("MMSI")
    .agg((F.max("LAT") - F.min("LAT") + F.max("LON") - F.min("LON")).alias("extension_grados"))
    .orderBy(F.desc("extension_grados"))
    .first()["MMSI"]
)
print("Buque de ejemplo:", MMSI_EJEMPLO)


def consulta_trayectoria(df):
    return (
        df.where(F.col("MMSI") == F.lit(MMSI_EJEMPLO))
          .where(F.col("day").between("2023-06-01", "2023-06-07"))
    )


def archivos_leidos_trayectoria(df):
    return (
        consulta_trayectoria(df)
        .select(F.col("_metadata.file_path").alias("archivo"))
        .distinct()
        .count()
    )


print("Archivos leidos por la trayectoria, Delta sin ZORDER:",
      archivos_leidos_trayectoria(spark.read.format("delta").load(RUTA_DELTA)))
print("Archivos leidos por la trayectoria, Delta + ZORDER  :",
      archivos_leidos_trayectoria(spark.read.format("delta").load(RUTA_DELTA_Z)))

Buque de ejemplo: 367641230
Archivos leidos por la trayectoria, Delta sin ZORDER: 6
Archivos leidos por la trayectoria, Delta + ZORDER  : 7


### 4.6 Costo asumido

El layout elegido favorece la consulta declarada. La celda anterior mide qué pasa con una consulta no declarada: la trayectoria de un buque, que filtra por `MMSI`. El buque de ejemplo pasa por la zona del tablero y es el que más se desplaza ese día.

Buque de ejemplo: `MMSI` 367641230. Archivos leídos:

| Consulta | Delta por `day` | Delta + `ZORDER (LAT, LON)` |
| --- | --- | --- |
| Tablero del operador (día y zona) | 1 de 7 (14,29%) | 1 de 60 (1,67%) |
| Trayectoria de un buque (7 días) | 6 de 7 (85,71%) | 7 de 60 (11,67%) |

**Lectura 1. La trayectoria es la única consulta que lee más archivos.** Pasa de 6 a 7 archivos. La partición por `day` obliga a abrir al menos un archivo por cada día en que el buque transmitió, y el Z-order por `LAT` y `LON` reparte la trayectoria de un buque que se desplaza entre varias zonas: en al menos un día cruzó dos archivos.

**Lectura 2. En volumen de datos la trayectoria también mejora.** Los 7 archivos después del Z-order suman unos 190 MB (27 MB en promedio), frente a unos 1,6 GB de los 6 archivos diarios completos de antes. El costo del layout sobre esta consulta es menor que el beneficio de la compactación en archivos más pequeños.

**El costo real está frente a otro layout.** Un `ZORDER BY (MMSI)` o un `CLUSTER BY (MMSI, BaseDateTime)` dejaría las posiciones de cada buque en muy pocos archivos, y la trayectoria sería la consulta rápida. A cambio, el tablero del operador volvería a leer casi todos los archivos del día, porque cada archivo tendría buques de todas las zonas. Un layout sirve a un patrón de acceso: elegir el layout es elegir qué patrón atiende rápido el sistema.

In [0]:
# 4.7 Variante opcional: agrupamiento líquido en lugar de partición.
# El agrupamiento líquido sustituye a la partición. Cambiar la bandera para compararlo.
EJECUTAR_LIQUID = False

if EJECUTAR_LIQUID:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.opt.ais_liquid")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.opt COMMENT 'Variantes de almacenamiento del requisito 4'")
    sql_liquid = (
        f"CREATE TABLE {CATALOG}.opt.ais_liquid "
        f"CLUSTER BY (day, LAT, LON) "
        f"AS SELECT * FROM {TABLE}"
    )
    spark.sql(sql_liquid)
    spark.sql(f"OPTIMIZE {CATALOG}.opt.ais_liquid")

    df_liquid = spark.table(f"{CATALOG}.opt.ais_liquid")
    detalle = spark.sql(f"DESCRIBE DETAIL {CATALOG}.opt.ais_liquid").collect()[0]
    print(f"Liquid: {humano(detalle['sizeInBytes'])} en {detalle['numFiles']} archivos")
    print("Archivos leidos por el tablero del operador:", archivos_leidos(df_liquid))
else:
    print("Variante de agrupamiento liquido desactivada.")

Variante de agrupamiento liquido desactivada.


# 5. Gobernanza

Todo el trabajo queda en el catálogo `ocean_watch` de Unity Catalog, separado por capas.

| Objeto | Tipo | Contenido | Se crea en |
| --- | --- | --- | --- |
| `ocean_watch` | Catálogo | Proyecto OceanWatch Analytics | 1.1 |
| `ocean_watch.raw` | Esquema | Datos tal como llegan de la fuente y tablas de referencia | 1.1 |
| `ocean_watch.raw.ais_raw` | Volume | `csv/` con los 7 CSV, `_manifest/` con la evidencia de integridad, `opt/` con las variantes de la sección 4 | 1.1 |
| `ocean_watch.raw.ais` | Tabla Delta | 60.533.559 posiciones AIS de la semana | 1.4 |
| `ocean_watch.raw.world_port_index` | Tabla Delta | Puertos del World Port Index | 1.8 |
| `ocean_watch.raw.ais_quality_report` | Tabla Delta | Diagnóstico consolidado de las reglas R1 a R9 | 2.6 |
| `ocean_watch.raw.ais_daily_profile` | Tabla Delta | Perfil de volumen por día | 2.6 |
| `ocean_watch.raw.vessel_type_catalog` | Tabla Delta | Catálogo de tipos de buque AIS | 3b |
| `ocean_watch.curated` | Esquema | Tablas derivadas de la limpieza | Anexo A |
| `ocean_watch.curated.ais_descarte_total` | Tabla Delta | Resultado del descarte total del Anexo A | Anexo A |

**Convenciones.**

- La capa `raw` no se modifica después de la carga. Toda transformación escribe en otro esquema.
- Todas las tablas llevan comentario de tabla. `ais` y `ais_descarte_total` llevan además comentario en cada columna y propiedades con la fuente y la cobertura.
- Los nombres de tabla se escriben siempre completos (`catalogo.esquema.tabla`), para no depender del catálogo por defecto de la sesión.

La celda siguiente agrega los comentarios que faltaban en `world_port_index` y consulta `information_schema` como evidencia: comentario de cada esquema y tabla, y cuántas columnas tienen comentario.

**Lectura del resultado.** Las 5 tablas de `raw` tienen comentario de tabla. `ais` (21 de 21), `world_port_index` (5 de 5) y `vessel_type_catalog` (2 de 2) tienen todas sus columnas comentadas. `ais_quality_report` (0 de 4) y `ais_daily_profile` (1 de 4) solo tienen comentario de tabla: queda como pendiente. El esquema `curated` y `ais_descarte_total` no aparecen porque se crean en el Anexo A, después de esta celda; el `DESCRIBE TABLE EXTENDED` de A.4 muestra su comentario de tabla y sus 21 columnas comentadas.

In [0]:
WPI_TABLE = f"{CATALOG}.{SCHEMA}.world_port_index"
WPI_COMMENTS = {
    "port_index": "Numero del puerto en el World Port Index",
    "port_name": "Nombre principal del puerto",
    "country_code": "Codigo de pais del World Port Index",
    "LAT": "Latitud del punto de referencia del puerto, grados decimales",
    "LON": "Longitud del punto de referencia del puerto, grados decimales",
}

spark.sql(f"""
    COMMENT ON TABLE {WPI_TABLE} IS
    'World Port Index (NGA): puertos del mundo con su punto de referencia. Se usa en el cruce de la pregunta 3d'
""")
for column, comment in WPI_COMMENTS.items():
    spark.sql(f"ALTER TABLE {WPI_TABLE} ALTER COLUMN `{column}` COMMENT '{comment}'")

display(spark.sql(f"""
    SELECT schema_name, comment
    FROM {CATALOG}.information_schema.schemata
    WHERE schema_name <> 'information_schema'
    ORDER BY schema_name
"""))

display(spark.sql(f"""
    SELECT
        t.table_schema,
        t.table_name,
        t.table_type,
        t.comment,
        count(c.column_name) AS columnas,
        count(c.comment) AS columnas_con_comentario
    FROM {CATALOG}.information_schema.tables AS t
    LEFT JOIN {CATALOG}.information_schema.columns AS c
        ON c.table_schema = t.table_schema AND c.table_name = t.table_name
    WHERE t.table_schema <> 'information_schema'
    GROUP BY ALL
    ORDER BY t.table_schema, t.table_name
"""))

schema_name,comment
default,Default schema (auto-created)
raw,"Capa raw: datos AIS tal como se publican en NOAA Marine Cadastre, sin limpieza"


table_schema,table_name,table_type,comment,columnas,columnas_con_comentario
raw,ais,MANAGED,"Posiciones AIS crudas de NOAA Marine Cadastre, 1 al 7 de junio de 2023. Una fila por mensaje de posicion, sin limpieza.",21,21
raw,ais_daily_profile,MANAGED,"Perfil de volumen diario de ocean_watch.raw.ais: posiciones, buques exactos y buques aproximados por dia.",4,1
raw,ais_quality_report,MANAGED,"Diagnostico de calidad de ocean_watch.raw.ais. Una fila por regla R1 a R9, con filas afectadas y porcentaje sobre el total. Insumo de la Entrega 2.",4,0
raw,vessel_type_catalog,MANAGED,"Catalogo de tipos de buque AIS (ITU-R M.1371, documentado por NOAA Marine Cadastre). Una fila por codigo de VesselType",2,2
raw,world_port_index,MANAGED,World Port Index (NGA): puertos del mundo con su punto de referencia. Se usa en el cruce de la pregunta 3d,5,5


# Anexo A. Limpieza por descarte de filas marcadas

Este anexo aplica la limpieza más simple posible: descartar toda fila que rompa una regla de la sección 2. El objetivo es medir el costo de esa decisión antes de tomarla en la Entrega 2.

### A.1 Las dos versiones del descarte

| Versión | Qué descarta | Criterio |
| --- | --- | --- |
| Descarte total | Toda fila marcada por cualquiera de las 9 reglas | Trata el valor de no disponible igual que el error |
| Descarte razonado | Solo las filas con error de contenido y los duplicados | Conserva el valor de no disponible y declara la cobertura |

La diferencia entre las dos está en R4 y R5a. R4 marca 33.535.628 filas con `Heading = 511`. R5a marca 39.624.930 filas sin número IMO asignado. Las dos condiciones describen la flota, y las dos aparecen en el estándar AIS como valores previstos.

### A.2 Qué mide este anexo

La celda siguiente calcula las filas que sobreviven a cada versión y persiste el resultado del descarte total en `ocean_watch.curated.ais_descarte_total`. La tabla queda con comentario de tabla y comentario por columna, de acuerdo con la práctica de gobernanza de 1.5.

In [0]:
# A.3 Definición de los dos criterios de descarte.
from pyspark.sql import Window

# Reutiliza las condiciones de la sección 2, por lo que R1 a R7 deben correr antes.

DESCARTE_TOTAL = (
    ~MMSI_VALIDO            # R1
    | COORD_FUERA           # R2
    | SOG_INVALIDO          # R3
    | (F.col("Heading") == 511)   # R4, valor de no disponible
    | IMO_AUSENTE           # R5a, valor de no disponible
    | IMO_IRREGULAR         # R5b
    | FILA_CORRUPTA         # R6
    | FUERA_DEL_DIA         # R7
)

DESCARTE_RAZONADO = (
    ~MMSI_VALIDO            # R1
    | COORD_FUERA           # R2
    | SOG_INVALIDO          # R3
    | IMO_IRREGULAR         # R5b
    | FILA_CORRUPTA         # R6
    | FUERA_DEL_DIA         # R7
)

# R8 y R9 se resuelven con deduplicación por la llave (MMSI, BaseDateTime).
ventana_llave = Window.partitionBy("MMSI", "BaseDateTime").orderBy("ingested_at")

ais_descarte_total = (
    ais.where(~DESCARTE_TOTAL)
    .withColumn("_fila", F.row_number().over(ventana_llave))
    .where(F.col("_fila") == 1)
    .drop("_fila")
)

ais_descarte_razonado = (
    ais.where(~DESCARTE_RAZONADO)
    .withColumn("_fila", F.row_number().over(ventana_llave))
    .where(F.col("_fila") == 1)
    .drop("_fila")
)

filas_total = ais_descarte_total.count()
filas_razonado = ais_descarte_razonado.count()

comparacion = spark.createDataFrame(
    [
        ("Tabla cruda", TOTAL_FILAS, 0),
        ("Descarte total", filas_total, TOTAL_FILAS - filas_total),
        ("Descarte razonado", filas_razonado, TOTAL_FILAS - filas_razonado),
    ],
    "version string, filas_conservadas long, filas_descartadas long",
).withColumn(
    "pct_conservado", F.round(F.col("filas_conservadas") * 100.0 / F.lit(TOTAL_FILAS), 2)
)

display(comparacion)

version,filas_conservadas,filas_descartadas,pct_conservado
Tabla cruda,60533559,0,100.0
Descarte total,14854796,45678763,24.54
Descarte razonado,60058373,475186,99.22


In [0]:
# A.4 Persistencia gobernada del descarte total.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.curated COMMENT 'Capa curated: tablas derivadas del perfilamiento de calidad de la seccion 2'")

TABLA_DESCARTE = f"{CATALOG}.curated.ais_descarte_total"

(
    ais_descarte_total.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .partitionBy("day")
    .saveAsTable(TABLA_DESCARTE)
)

COMENTARIO_TABLA = (
    "Resultado de la limpieza por descarte total del Anexo A. Conserva las filas de "
    "ocean_watch.raw.ais que no rompen ninguna regla R1 a R7, con deduplicacion por "
    "(MMSI, BaseDateTime). Descarta tambien los valores de no disponible de R4 y R5a, "
    "por lo que no sirve como base de analisis general. Ver Anexo A."
)
spark.sql(f"COMMENT ON TABLE {TABLA_DESCARTE} IS '{COMENTARIO_TABLA}'")

# Reutiliza los comentarios de columna de 1.5 para conservar la misma documentacion.
for columna, comentario in COLUMN_COMMENTS.items():
    if columna in ais_descarte_total.columns:
        spark.sql(
            f"ALTER TABLE {TABLA_DESCARTE} ALTER COLUMN `{columna}` "
            f"COMMENT '{comentario}'"
        )

display(spark.sql(f"DESCRIBE TABLE EXTENDED {TABLA_DESCARTE}"))

col_name,data_type,comment
MMSI,string,"Maritime Mobile Service Identity. Identificador del buque, deberia tener 9 digitos"
BaseDateTime,timestamp,Fecha y hora UTC del reporte de posicion
LAT,double,"Latitud en grados decimales, rango valido [-90, 90]"
LON,double,"Longitud en grados decimales, rango valido [-180, 180]"
SOG,double,Speed over ground: velocidad sobre el fondo en nudos
COG,double,"Course over ground: rumbo sobre el fondo en grados [0, 360)"
Heading,double,"Rumbo de la proa en grados [0, 359]. 511 significa no disponible"
VesselName,string,Nombre del buque reportado por el transpondedor
IMO,string,"Numero IMO con prefijo, por ejemplo IMO9074729. Vacio o IMO0000000 si no se reporta"
CallSign,string,Indicativo de llamada de radio


### A.5 Lectura del resultado

| Versión | Filas conservadas | Filas descartadas | % conservado |
| --- | --- | --- | --- |
| Tabla cruda | 60.533.559 | 0 | 100% |
| Descarte total | 14.854.796 | 45.678.763 | 24,54% |
| Descarte razonado | 60.058.373 | 475.186 | 99,22% |

El descarte total elimina 3 de cada 4 filas. El descarte razonado elimina el 0,78%, que coincide con la cota de error de contenido de la Ficha 2.6 (R1, R3 y R5b) más los duplicados.

**Por qué el descarte total no sirve como base de análisis.** R4 marca el 55,40% de las filas y R5a el 65,46%. Las dos reglas miden valores que el estándar AIS prevé. Un descarte que las incluye elimina a casi toda la flota de Clase B, que aporta el 32,96% de las posiciones y el 87,7% de los casos de `Heading = 511`. El resultado deja de representar el tráfico marítimo real.

**Sesgo que introduce el descarte total.** La flota que sobrevive es la que lleva equipo Clase A con número IMO asignado, es decir, buques de carga y pasaje de tráfico internacional. Las respuestas de la sección 3 cambiarían de sentido. La pregunta 3b perdería los códigos 31 y 37, que juntos aportan el 51,27% de las posiciones.

**Criterio recomendado para la Entrega 2.** El descarte razonado elimina solo el error de contenido y los duplicados. Las columnas con valor de no disponible se conservan, y cada análisis declara la cobertura que usa. Ese enfoque conserva el volumen y mantiene la trazabilidad frente a la tabla cruda.

**Decisión técnica 1. La capa `raw` no se modifica.** `ocean_watch.raw.ais` conserva el dato tal como llega del origen, según el comentario del esquema de 1.1. Las tablas limpias viven en `ocean_watch.curated`.

**Decisión técnica 2. Deduplicación con función de ventana.** `row_number` sobre la partición (`MMSI`, `BaseDateTime`) conserva una fila por llave. `dropDuplicates` sobre las mismas dos columnas da el mismo conteo y no permite elegir cuál fila queda. La función de ventana deja ese criterio explícito.

**Decisión técnica 3. La misma partición que la capa cruda.** La tabla se escribe particionada por `day`, de acuerdo con la evidencia de la sección 4. Los comentarios de columna se copian de `COLUMN_COMMENTS` de 1.5 para no duplicar la documentación.